# Quantum calibration of local-volatility surfaces from sparse option quotes

**Run all cells on a CPU runtime (Google Colab or a local Jupyter). No input files have to be uploaded.**

This notebook accompanies the paper of the same title (IEEE SSCI 2027, CIFEr). It builds the quantum circuits, runs the **calibration experiment** in which the coefficients are updated from finite-shot measurement counts, and writes the figures, tables, CSV files, measurement logs, executable circuits and the technical supplement.

### What the experiments test

1. **Financial task**: estimate the local volatility of the pricing equation from prices at several strikes and maturities.
2. **Role of the quantum circuit**: propagate prices for candidate coefficients and turn the difference from the observed prices into a measurement probability. A classical search updates the coefficients from the measured values.
3. **How to read the comparisons**: agreement with classical prices of the same model, the difference from surfaces of other models, measurement precision and circuit cost.

The smile and skew examples compare exact propagation, finite differences and LCHS at common, classically calibrated coefficients. The calibration from quantum measurements is tested on a separate grid of four nodes.
Implied volatility (IV) restates a price in the market's volatility quotation; local volatility is the coefficient of the model that produces the prices. The figures keep the two apart.
One basis point (bp) of price error is 0.0001 of the normalised price.

### Scope of execution

| Experiment | In this notebook | What it shows |
|---|---|---|
| Measured multi-expiry calibration | Rerun by default: 4 price nodes, 4 coefficients, 2 maturities | Logical circuits, measured counts and the behaviour of the calibration loop |
| Four-maturity smile and skew | Calibrated and computed by default: 128 nodes, 3 coefficients per interval | Financial shapes and the forward approximations at the same coefficients |
| Financial comparison on larger grids | Stored results are audited and redrawn | Classical solvers, matrix LCHS, AH and PINN |
| Fixed-coefficient consistency (Fig. 4) | Four methods re-evaluated at stored coefficients | Price error and butterfly prices as the number of terms grows |
| Full recomputation of the large comparison | A separate notebook is written at the end | Can be rerun independently when needed |

`paper` runs two shot budgets with three seeds each. Allow tens of minutes or more on a CPU.
`smoke` checks the execution path and does not reproduce the numbers of the paper.
`precision` is an additional search with more quadrature terms. It can take **several hours** and is not an experiment of the paper.
Nothing is sent to hardware: execution is logical gate simulation on Qiskit Aer with finite sampling.


In [ ]:
# Run settings. "paper" is the configuration of the paper.
PROFILE = "paper"  # "smoke", "pilot", "paper", "precision"
BUILD_PDF = True
INSTALL_DEPENDENCIES = True
DOWNLOAD_ZIP = False  # Set True and rerun the last cell to download the archive in Colab.
import os, sys, subprocess, importlib.metadata, pathlib
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
OUTPUT_ROOT = "/content/quantum_surface_run" if pathlib.Path("/content").exists() else "quantum_surface_run"
# Dependency versions of the validated environment. Use a CPU runtime.
packages = ["numpy==2.3.5", "scipy==1.17.0", "pandas==2.2.3",
            "matplotlib==3.10.8", "qiskit==2.5.2", "qiskit-aer==0.17.2", "pypdf==5.9.0"]
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*packages])
assert PROFILE in {"smoke","pilot","paper","precision"}
print("Profile:",PROFILE,"| Output:",OUTPUT_ROOT)


## 1. Prepare the sources and the stored data

The next cell holds the source modules and the TeX of the supplement. The Python files under `src/` can be read and edited as they are.
The compressed data belong to the archived financial comparison only. Their hash and extraction paths are checked, and the Python code archived with them is not executed.


In [ ]:
# @title Embedded sources and archived benchmark (expand to inspect)
SOURCES = {'audit_results.py': '"""Independent audit and publication tables for the supplied completed study.\nRun from the extracted review bundle: python audit_results.py --input evidence --out revised\nNo calibration is rerun and no uploaded code is executed.\n"""\nfrom pathlib import Path\nfrom itertools import product\nimport argparse, hashlib, json, shutil, io\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\nLABEL={"expm":"Exact expm","fd":"FD + adjoint","lchs-beta":"LCHS beta","lchs-simple":"LCHS simple",\n       "AH":"Andreasen--Huge","PINN-fixed":"PINN fixed","PINN-Wam":"PINN adaptive"}\nORDER=list(LABEL)\n\ndef read_json(path): return json.loads(Path(path).read_text())\ndef write_json(path,obj): Path(path).write_text(json.dumps(obj,indent=2,allow_nan=False))\ndef interp(K,query,C): return np.interp(np.asarray(query),K,np.asarray(C))\n\ndef audit_run(row,result,data):\n    N=int(row["N"]); K=np.exp(np.linspace(-1.2,1.2,N+2)); C=np.asarray(result["C"])\n    assert C.shape==(2,N+2) and np.isfinite(C).all()\n    quote_res=[]; test_res=[]; model_tests=[]; reference_tests=[]\n    for s,(q,test) in enumerate(zip(data["quotes"],data["tests"])):\n        train_k=np.asarray(q["k"]); test_k=np.asarray(test["k"])\n        assert not np.any(np.isclose(test_k[:,None],train_k[None,:],rtol=0,atol=1e-10))\n        qp=np.asarray(result["quote_predictions"][s]) if "quote_predictions" in result else interp(K,train_k,C[s])\n        hp=np.asarray(result["test_predictions"][s]) if "test_predictions" in result else interp(K,test_k,C[s])\n        quote_res.extend(qp-np.asarray(q["mid"])); test_res.extend(hp-np.asarray(test["price"]))\n        model_tests.extend(hp); reference_tests.extend(test["price"])\n    dm=np.diff(K)[:-1]; dp=np.diff(K)[1:]\n    b=(dm*C[:,2:]+dp*C[:,:-2])/(dm+dp)-C[:,1:-1]\n    cal=np.diff(np.vstack([np.maximum(1-K,0),C]),axis=0)\n    band=(K[:-2]>=.7)&(K[2:]<=1.4); calband=(K>=.7)&(K<=1.4)\n    audit=dict(rmse_quotes=float(np.sqrt(np.mean(np.square(quote_res)))),\n        rmse_heldout=float(np.sqrt(np.mean(np.square(test_res)))),\n        butterfly_violation=float(max(0,-b.min())),calendar_violation=float(max(0,-cal.min())),\n        density_violation=float(max(0,-(2*b/(dm*dp)).min())),\n        bfly_count=int((b < -1e-8).sum()),cal_count=int((cal < -1e-8).sum()),\n        band_bfly_count=int((b[:,band] < -1e-8).sum()),band_cal_count=int((cal[:,calband] < -1e-8).sum()),\n        band_bfly_violation=float(max(0,-b[:,band].min())),band_cal_violation=float(max(0,-cal[:,calband].min())),\n        invalid_model_price_count=int((~np.isfinite(model_tests)).sum()),\n        maximum_metric_discrepancy=0.)\n    for name in ["rmse_quotes","rmse_heldout","butterfly_violation","calendar_violation","density_violation","bfly_count","cal_count"]:\n        discrepancy=abs(audit[name]-float(row[name]))\n        audit["maximum_metric_discrepancy"]=max(audit["maximum_metric_discrepancy"],discrepancy)\n        assert np.isclose(audit[name],row[name],rtol=2e-8,atol=2e-11),(name,audit[name],row[name])\n    if "C_exact" in result:\n        exact=np.asarray(result["C_exact"]); delta=np.max(np.abs(C-exact),axis=1)\n        be=(dm*exact[:,2:]+dp*exact[:,:-2])/(dm+dp)-exact[:,1:-1]\n        ce=np.diff(np.vstack([np.maximum(1-K,0),exact]),axis=0)\n        audit["exact_bf_violation"]=float(max(0,-be.min()))\n        audit["exact_cal_violation"]=float(max(0,-ce.min()))\n        assert audit["exact_bf_violation"]<1e-8 and audit["exact_cal_violation"]<1e-8\n        cb=delta+np.r_[0,delta[:-1]]\n        assert np.all(b>=-2*delta[:,None]-2e-10)\n        assert np.all(cal>=-cb[:,None]-2e-10)\n        audit["conditional_certificate_pass"]=True\n        audit["measured_price_error_inf"]=float(delta.max())\n    return audit\n\ndef figure(fig,out,name,source,caption):\n    for ax in fig.axes:\n        if ax.get_label()!="<colorbar>": ax.grid(alpha=.15,linewidth=.5)\n    for suffix in ["pdf","png"]:\n        stream=io.BytesIO()\n        fig.savefig(stream,format=suffix,bbox_inches="tight",pad_inches=.05,dpi=240)\n        content=stream.getvalue()\n        if suffix=="pdf": assert content.rstrip().endswith(b"%%EOF")\n        destination=out/"figures"/(name+"."+suffix)\n        temporary=destination.with_suffix(destination.suffix+".tmp")\n        temporary.write_bytes(content);temporary.replace(destination)\n    write_json(out/"figures"/(name+".json"),dict(source=source,caption=caption))\n    plt.close(fig)\n\ndef run(root,out):\n    root=Path(root); out=Path(out)\n    for name in ["tables","figures","audit"]: (out/name).mkdir(parents=True,exist_ok=True)\n    d=pd.read_csv(root/"results.csv"); m=read_json(root/"manifest.json"); cfg=m["config"]\n    assert cfg["profile"]=="paper" and cfg["fingerprint"]=="9684a7f0bec7", "This review is specific to the supplied paper run."\n    hashes={n:hashlib.sha256((root/"source"/n).read_bytes()).hexdigest()==h for n,h in m["source_sha256"].items()}\n    assert all(hashes.values())\n    expected=set()\n    for truth,noise in product(["in","out"],[0.,5.]):\n        for ds in ([101] if noise==0 else cfg["noise_seeds"]):\n            dataset=f"{truth}_bp{noise:g}_d{ds}"\n            for n in cfg["ns"]:\n                methods=ORDER if n==6 else ORDER[:4]\n                for method in methods:\n                    for seed in ([0] if method=="AH" else cfg["seeds"]): expected.add((dataset,n,method,seed))\n    observed=set(zip(d.dataset,d.n,d.method,d.seed)); assert len(observed)==len(d)\n    failed=read_json(root/"failures.json")\n    failures={(f["dataset"],int(f["key"].split("_n")[-1].split("_")[0]),f["method"],int(f["key"].rsplit("_s",1)[1])) for f in failed}\n    assert not observed.intersection(failures) and expected==observed|failures\n    raw={f.stem:read_json(f) for f in (root/"raw").glob("*.json")}\n    records=[]\n    for row in d.to_dict("records"):\n        key=f\'{row["dataset"]}_n{row["n"]}_{row["method"]}_s{row["seed"]}\'\n        payload=read_json(root/"checkpoints"/(key+".json"))\n        for name in ["rmse_quotes","rmse_heldout","opt_s","compile_warm_s"]:\n            assert np.isclose(row[name],payload["row"][name],rtol=1e-12,atol=1e-12)\n        check=audit_run(row,payload["result"],raw[row["dataset"]])\n        records.append(dict(key=key,dataset=row["dataset"],method=row["method"],n=row["n"],seed=row["seed"],**check))\n    checks=pd.DataFrame(records); checks.to_csv(out/"audit"/"independent_checks.csv",index=False)\n    # Audit outcomes are statements about nodal tests on their stated domains.\n    arb=checks.groupby(["method","n"]).agg(runs=("key","size"),bf_fail=("bfly_count",lambda x:int((x>0).sum())),\n         cal_fail=("cal_count",lambda x:int((x>0).sum())),bf_max=("butterfly_violation","max"),cal_max=("calendar_violation","max"),\n         band_bf_fail=("band_bfly_count",lambda x:int((x>0).sum())),band_cal_fail=("band_cal_count",lambda x:int((x>0).sum())),\n         band_bf_max=("band_bfly_violation","max"),band_cal_max=("band_cal_violation","max")).reset_index()\n    arb.to_csv(out/"audit"/"arbitrage_by_method_grid.csv",index=False)\n    ds=d.groupby(["truth","noise_bp","n","method","dataset"],as_index=False).agg(\n         heldout=("rmse_heldout","mean"),quote=("rmse_quotes","mean"),opt_s=("opt_s","mean"),warm_s=("compile_warm_s","mean"),runs=("seed","size"))\n    summary=ds.groupby(["truth","noise_bp","n","method"]).agg(heldout=("heldout","mean"),heldout_sd=("heldout","std"),\n         quote=("quote","mean"),opt_s=("opt_s","mean"),warm_s=("warm_s","mean"),datasets=("dataset","size"),runs=("runs","sum")).reset_index()\n    ds.to_csv(out/"audit"/"dataset_means_verified.csv",index=False); summary.to_csv(out/"audit"/"summary_verified.csv",index=False)\n    iv=d.groupby(["method","n"]).agg(valid=("iv_valid","sum"),total=("iv_total","sum")).reset_index()\n    iv["invalid"]=iv.total-iv.valid; iv.to_csv(out/"audit"/"iv_coverage.csv",index=False)\n    b=pd.read_csv(root/"error_budget.csv"); assert len(b)==90 and b.certificate_pass.all()\n    # A quadrature certificate is conditional and is not an arbitrage pass.\n    b.to_csv(out/"audit"/"kernel_budget_source.csv",index=False)\n    q=read_json(root/"quantum_resources.json"); qo=pd.read_csv(root/"quantum_objective.csv")\n    oracle=float(np.max(np.abs(qo.circuit_block_real+1j*qo.circuit_block_imag-qo.normalised_loss)))\n    assert abs(oracle-q["objective_block_error"])<1e-14\n    outcase=summary[(summary.n==6)&(summary.truth=="out")&(summary.noise_bp==5)].set_index("method")\n    stats=dict(planned_runs=len(expected),completed_runs=len(d),failed_runs=len(failed),source_hashes_match=hashes,\n          unique_run_keys=True,all_expected_runs_accounted=True,\n          maximum_recomputed_metric_discrepancy=float(checks.maximum_metric_discrepancy.max()),\n          conditional_calibration_certificates=int((checks.conditional_certificate_pass==True).sum()),\n          lchs_conditional_calibration_certificates=int(((checks.conditional_certificate_pass==True)&checks.method.str.startswith("lchs")).sum()),\n          exact_re_evaluation_bf_max=float(checks.loc[checks.method.str.startswith("lchs"),"exact_bf_violation"].max()),\n          exact_re_evaluation_cal_max=float(checks.loc[checks.method.str.startswith("lchs"),"exact_cal_violation"].max()),\n          forward_budget_certificates=int(b.certificate_pass.sum()),\n          lchs_beta_raw_bf_fail=int(((checks.method=="lchs-beta")&(checks.bfly_count>0)).sum()),\n          lchs_beta_raw_cal_fail=int(((checks.method=="lchs-beta")&(checks.cal_count>0)).sum()),\n          classical_beta_to_fd_opt_time_ratio=float(outcase.loc["lchs-beta","opt_s"]/outcase.loc["fd","opt_s"]),\n          classical_beta_to_expm_opt_time_ratio=float(outcase.loc["lchs-beta","opt_s"]/outcase.loc["expm","opt_s"]),\n          oracle_block_error=oracle,market_status=read_json(root/"market_status.json")["status"],\n          scope="Independent audit of supplied results; original optimisation was not rerun")\n    write_json(out/"audit"/"audit_summary.json",stats)\n    # Main comparison: all seven methods, three noise datasets, dataset-level SD.\n    rows=[r"\\begin{table}[t]\\centering\\caption{Out-of-family, 5 bp noise, $N=64$. Price errors are normalised bp. Test: mean $\\pm$ sample SD across three datasets after averaging initialisations. Time is mean classical optimisation seconds.}",\n          r"\\label{tab:bench}\\scriptsize\\begin{tabular}{lrrrr}\\toprule Method & Fit & Test & Time & Runs\\\\\\midrule"]\n    for method in ORDER:\n        r=outcase.loc[method]\n        rows.append(f"{LABEL[method]} & {r.quote*1e4:.2f} & ${r.heldout*1e4:.2f}\\\\pm{r.heldout_sd*1e4:.2f}$ & {r.opt_s:.3g} & {int(r.runs)}"+r"\\\\")\n    rows.append(r"\\bottomrule\\end{tabular}\\end{table}"); (out/"tables"/"benchmark.tex").write_text("\\n".join(rows))\n    rows=[r"\\begin{table}[t]\\centering\\caption{Raw beta-LCHS audits at price tolerance $10^{-8}$. Full-domain violation maxima are normalised bp. The last column counts butterfly-failing runs within $[0.7,1.4]$.}",\n          r"\\label{tab:arb}\\scriptsize\\begin{tabular}{rrrrrr}\\toprule $N$ & BF fail & Cal. fail & BF max & Cal. max & Band fail\\\\\\midrule"]\n    for r in arb[arb.method=="lchs-beta"].itertuples():\n        rows.append(f"{2**r.n} & {r.bf_fail}/{r.runs} & {r.cal_fail}/{r.runs} & {r.bf_max*1e4:.4f} & {r.cal_max*1e4:.4f} & {r.band_bf_fail}/{r.runs}"+r"\\\\")\n    rows.append(r"\\bottomrule\\end{tabular}\\end{table}");(out/"tables"/"arbitrage_audit.tex").write_text("\\n".join(rows))\n    # Full-case appendix table prevents one-case selection from hiding results.\n    rows=[r"\\begin{table}[htbp]\\centering\\caption{Held-out price RMSE at $N=64$ (normalised bp). Noisy columns average initialisations within each dataset, then datasets. The in-family AH noisy entry uses only two valid datasets.}\\small",\n          r"\\begin{tabular}{lrrrr}\\toprule Method & In, clean & In, noisy & Out, clean & Out, noisy\\\\\\midrule"]\n    for method in ORDER:\n        values=[]\n        for truth,noise in [("in",0),("in",5),("out",0),("out",5)]:\n            r=summary[(summary.n==6)&(summary.method==method)&(summary.truth==truth)&(summary.noise_bp==noise)].iloc[0]\n            values.append(f"{1e4*r.heldout:.3f}"+(f" $\\\\pm$ {1e4*r.heldout_sd:.3f}" if noise else ""))\n        rows.append(LABEL[method]+" & "+" & ".join(values)+r"\\\\")\n    rows.append(r"\\bottomrule\\end{tabular}\\end{table}");(out/"tables"/"all_cases.tex").write_text("\\n".join(rows))\n    # Pair the two feasible noisy in-family datasets for all seven methods.\n    paired=ds[(ds.n==6)&(ds.truth=="in")&(ds.noise_bp==5)&ds.dataset.isin(["in_bp5_d101","in_bp5_d202"])]\n    paired=paired.groupby("method").agg(mean=("heldout","mean"),sd=("heldout","std"),datasets=("dataset","size"))\n    paired.to_csv(out/"audit"/"in_noisy_complete_case.csv")\n    rows=[r"\\begin{table}[htbp]\\centering\\caption{Paired feasible in-family noisy datasets (101 and 202), $N=64$. Held-out RMSE in normalised bp; sample SD over two datasets. Dataset 303 remains a separately reported failed-input stress case.}\\small",\n          r"\\begin{tabular}{lrr}\\toprule Method & Mean & Dataset SD\\\\\\midrule"]\n    for method in ORDER:\n        r=paired.loc[method];rows.append(f"{LABEL[method]} & {r[\'mean\']*1e4:.3f} & {r.sd*1e4:.3f}"+r"\\\\")\n    rows.append(r"\\bottomrule\\end{tabular}\\end{table}");(out/"tables"/"paired_in_family.tex").write_text("\\n".join(rows))\n    # Scientific figure contracts: explicit method identities, two-root palette,\n    # neutral baselines, published units and all specified dataset replicates.\n    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":8,"axes.titlesize":8,"legend.fontsize":7,\n        "xtick.labelsize":7,"ytick.labelsize":7,"pdf.fonttype":42,"axes.spines.top":False,"axes.spines.right":False})\n    fig,ax=plt.subplots(figsize=(3.5,2.25),layout="constrained")\n    for i,method in enumerate(ORDER):\n        r=outcase.loc[method]; color="#CB702C" if method=="lchs-beta" else "#236C9C" if method=="lchs-simple" else "#59636B"\n        ax.errorbar(r.heldout*1e4,i,xerr=r.heldout_sd*1e4,fmt="o",color=color,capsize=2,ms=4)\n    ax.set_yticks(range(len(ORDER)),[LABEL[x].replace("--","-") for x in ORDER]);ax.invert_yaxis()\n    ax.set(xlabel="Held-out RMSE (normalised bp)",title="N=64 / out-of-family / 5 bp noise",xlim=(0,None))\n    figure(fig,out,"heldout_comparison",["results.csv","audit/dataset_means_verified.csv"],"Mean and sample SD over three data seeds after averaging five initialisations (AH deterministic).")\n    fig,axs=plt.subplots(1,2,figsize=(3.5,2.1),layout="constrained")\n    for n,marker in [(5,"o"),(6,"s"),(7,"^")]:\n        p=b[(b.n==n)&(b.kernel=="beta")]\n        for cutoff,linestyle in [(16.,":"),(32.,"--"),(64.,"-")]:\n            v=p[p.cutoff==cutoff].sort_values("terms")\n            if n==7: axs[0].loglog(v.terms,v.price_error_inf,marker=marker,ms=3,ls=linestyle,color="#236C9C",label=f"K={cutoff:g}")\n        v=p[p.cutoff==64.].sort_values("terms")\n        axs[1].loglog(v.terms,v.price_error_inf,marker=marker,ms=3,color={5:"#60676C",6:"#236C9C",7:"#CB702C"}[n],label=f"N={2**n}")\n    from matplotlib.ticker import NullFormatter\n    for ax in axs:\n        ax.set_xticks([256,1024,4096],["256","1024","4096"]);ax.xaxis.set_minor_formatter(NullFormatter());ax.set_xlabel("LCU terms J");ax.legend()\n    axs[0].set(title="N=128, beta kernel",ylabel="Max price error");axs[1].set(title="Cutoff K=64",ylabel="Max price error")\n    figure(fig,out,"kernel_resolution",["error_budget.csv"],"All beta-kernel cutoffs at N=128 and fixed-cutoff spatial-grid comparison. Designs differ across spatial grids, as in the input protocol.")\n    # Example selected by a fixed source key, not the final loop iteration.\n    key="in_bp0_d101_n7_lchs-beta_s0"; r=read_json(root/"checkpoints"/(key+".json"))["result"]\n    K=np.exp(np.linspace(-1.2,1.2,130)); C=np.asarray(r["C"]); E=np.asarray(r["C_exact"])\n    dm=np.diff(K)[:-1];dp=np.diff(K)[1:]\n    bf=lambda A:((dm*A[:,2:]+dp*A[:,:-2])/(dm+dp)-A[:,1:-1])\n    fig,axs=plt.subplots(2,1,figsize=(3.5,2.8),layout="constrained",sharex=True)\n    axs[0].plot(K,(C[1]-E[1])*1e4,color="#CB702C",lw=.9);axs[0].axhline(0,color=".5",lw=.7)\n    axs[0].set(ylabel="Raw minus exact (bp)",title="N=128 / in-family / no noise / seed 0 / T=1")\n    axs[1].plot(K[1:-1],bf(C)[1]*1e4,color="#CB702C",lw=.9,label="Raw beta LCHS")\n    axs[1].plot(K[1:-1],bf(E)[1]*1e4,color="#59636B",ls="--",lw=.9,label="Exact re-evaluation")\n    axs[1].axhline(0,color=".5",lw=.7);axs[1].set(xlabel="Normalised strike k",ylabel="Price butterfly (bp)");axs[1].legend()\n    for ax in axs: ax.axvspan(.7,1.4,color=".9",alpha=.5);ax.set_xlim(K[0],K[-1])\n    figure(fig,out,"raw_output_audit",[f"checkpoints/{key}.json"],"Fixed design, second maturity. Shading marks the quote/test strike band. No clipping or repair is applied.")\n    print(json.dumps(stats,indent=2))\n    return stats\n\nif __name__=="__main__":\n    parser=argparse.ArgumentParser();parser.add_argument("--input",required=True);parser.add_argument("--out",required=True)\n    args=parser.parse_args();run(args.input,args.out)\n', 'consistency_example.py': '"""Fixed-coefficient financial diagnostic of a retained calibration stress case.\n\nNo optimisation, new quotes, circuit measurements, or smoothing are performed.\nThe same fitted first-expiry coefficients are propagated with the old LCHS\nbudget and a doubled budget; exact and backward Euler are classical references.\n"""\nfrom pathlib import Path\nimport json,hashlib,time\nimport numpy as np\nimport pandas as pd\nfrom surface_example import make_grid,add_shift,advance\nfrom surface_model import save_json\n\nKEY=\'out_bp0_d101_n7_lchs-beta_s0\'\nT=182/365\n\ndef run(evidence,out):\n    evidence=Path(evidence);out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    checkpoint=evidence/\'checkpoints\'/f\'{KEY}.json\'\n    data=json.loads(checkpoint.read_text());r=data[\'result\'];xi=np.array(r[\'hats\'])[0]\n    # Reconstruct the archived four-cosine basis on its fitted log-strike grid.\n    g=make_grid(128);x=g[\'x\'][1:-1]\n    z=(x[None,:]-np.array([-.6,-.2,.2,.6])[:,None])/.45\n    g[\'phi\']=.04*(1+np.cos(np.pi*np.clip(z,-1,1)))*(abs(z)<=1)\n    add_shift(g)\n    records=[];prices={};timings={};imaginary={}\n    for method,J in [(\'exact\',0),(\'fd\',0),(\'lchs\',2048),(\'lchs\',4096)]:\n        label=method if not J else f\'lchs_{J}\'\n        start=time.perf_counter();v=advance(g,xi,T,g[\'v0\'],method,J or 2048,48.)\n        timings[label]=time.perf_counter()-start;imaginary[label]=float(np.max(abs(np.imag(v))))\n        assert imaginary[label]<1e-9\n        c=g[\'G\'].copy();c[1:-1]+=np.real(v);prices[label]=c\n    exact_diff=float(np.max(abs(prices[\'exact\']-np.array(r[\'C_exact\'])[0])))\n    lchs_diff=float(np.max(abs(prices[\'lchs_2048\']-np.array(r[\'C\'])[0])))\n    assert exact_diff<1e-9 and lchs_diff<1e-9,(exact_diff,lchs_diff)\n    k=g[\'K\'];dm=np.diff(k)[:-1];dp=np.diff(k)[1:]\n    interior=(k[:-2]>=.7)&(k[2:]<=1.4);band=(k>=.7)&(k<=1.4)\n    checks={}\n    for method,c in prices.items():\n        bf=(dm*c[2:]+dp*c[:-2])/(dm+dp)-c[1:-1]\n        delta=float(np.max(abs(c-prices[\'exact\'])))\n        assert bf.min()>=-2*delta-1e-9\n        for i in range(len(k)):\n            records.append(dict(method=method,T=T,k=k[i],price=c[i],exact_price=prices[\'exact\'][i],\n                error_bp=(c[i]-prices[\'exact\'][i])*1e4,\n                butterfly_bp=bf[i-1]*1e4 if 0<i<len(k)-1 else np.nan,\n                in_quote_band=bool(band[i]),complete_butterfly_in_band=bool(interior[i-1]) if 0<i<len(k)-1 else False))\n        checks[method]=dict(max_inband_price_error_bp=float(np.max(abs(c[band]-prices[\'exact\'][band]))*1e4),\n            min_inband_butterfly_bp=float(bf[interior].min()*1e4),\n            inband_butterfly_failures=int(np.sum(bf[interior]<-1e-8)),\n            global_butterfly_failures=int(np.sum(bf<-1e-8)),\n            global_max_price_error_bp=delta*1e4)\n    pd.DataFrame(records).to_csv(out/\'consistency_predictions.csv\',index=False)\n    result=dict(checkpoint=KEY,checkpoint_sha256=hashlib.sha256(checkpoint.read_bytes()).hexdigest(),\n        scope=\'First-expiry forward re-evaluation at the same retained LCHS-fitted coefficients; no recalibration or quantum hardware execution\',\n        selection=\'Clean outside-model target, N=128, optimiser start 0. This stress case attains the archived maximum quoted-band butterfly deficit; it is not an average case.\',\n        N=128,T=T,coefficients=xi.tolist(),J=[2048,4096],cutoff=48.,beta=.8,fd_steps=1024,shift=g[\'shift\'],\n        exact_archive_max_difference=exact_diff,lchs_archive_max_difference=lchs_diff,\n        max_imaginary_part=imaginary,forward_seconds=timings,metrics=checks)\n    save_json(out/\'consistency_validation.json\',result)\n    return result\n\nif __name__==\'__main__\':\n    import sys\n    print(json.dumps(run(sys.argv[1],sys.argv[2]),indent=2))\n', 'experiment_figures.py': '"""Source-backed paper figures and TeX values for the new circuit experiments."""\nfrom pathlib import Path\nimport ast, json, io\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom surface_model import (grid,data_for,prices,observation,implied_vol,TRUTH,\n                           financial_metrics,save_json)\n\nBLUE=\'#286F9F\'; ORANGE=\'#CB702C\'; INK=\'#252A2D\'; GREY=\'#737C82\'\n\ndef finish(fig,out,name,sources):\n    out=Path(out);(out/\'figures\').mkdir(parents=True,exist_ok=True)\n    for fmt in [\'pdf\',\'png\']:\n        buf=io.BytesIO();fig.savefig(buf,format=fmt,dpi=210,bbox_inches=\'tight\',facecolor=\'white\')\n        payload=buf.getvalue()\n        if fmt==\'pdf\':assert payload.rstrip().endswith(b\'%%EOF\')\n        path=out/\'figures\'/f\'{name}.{fmt}\';tmp=path.with_suffix(path.suffix+\'.tmp\')\n        tmp.write_bytes(payload);tmp.replace(path)\n    save_json(out/\'figures\'/f\'{name}.json\',dict(sources=sources))\n    plt.close(fig)\n\ndef generate(run,out):\n    run=Path(run);out=Path(out);(out/\'tables\').mkdir(parents=True,exist_ok=True)\n    cfg=json.loads((run/\'configuration.json\').read_text())\n    r=pd.read_csv(run/\'calibration_results.csv\');quant=r[r.method==\'quantum_counts\']\n    low,high=cfg[\'shots\'][0],cfg[\'shots\'][-1]\n    plt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'font.size\':8,\'axes.labelsize\':8,\n        \'axes.titlesize\':9,\'legend.fontsize\':7,\'axes.spines.top\':False,\n        \'axes.spines.right\':False,\'pdf.fonttype\':42})\n    problem=data_for(grid());g=problem[\'g\'];k=np.linspace(.7,1.4,121);W=observation(g,k)\n    target=prices(g,TRUTH);mat=r[r.method==\'lchs\'].iloc[0]\n    chosen=[(\'Same-grid target\',TRUTH,\'exact\',INK,\'-\'),\n            (\'Matrix LCHS\',np.array(ast.literal_eval(mat.parameters)).reshape(2,2),\'lchs\',ORANGE,\'--\')]\n    for B,style,name in [(low,\':\',\'Measured: lower shots\'),(high,\'-.\',\'Measured: higher shots\')]:\n        subset=quant[(quant.shots==B)&(quant.seed==cfg[\'seeds\'][0])]\n        if len(subset):chosen.append((name,np.array(ast.literal_eval(subset.iloc[0].parameters)).reshape(2,2),\'lchs\',BLUE,style))\n    fig,axs=plt.subplots(1,3,figsize=(7.15,2.45),gridspec_kw={\'width_ratios\':[1,1,1.1]})\n    records=[]\n    for label,xi,method,col,ls in chosen:\n        p=prices(g,xi,method=method,terms=cfg[\'terms\'],cutoff=cfg[\'cutoff\'])\n        for s,T in enumerate(problem[\'times\']):\n            pred=(W@p[s]).real;iv=np.array([implied_vol(a,T,b) for a,b in zip(k,pred)])*100\n            axs[s].plot(k,iv,color=col,ls=ls,lw=1.35,label=label)\n            records.extend(dict(method=label,T=T,k=a,price=b,iv_pct=c,selected_seed=cfg[\'seeds\'][0])\n                           for a,b,c in zip(k,pred,iv))\n    for s,T in enumerate(problem[\'times\']):\n        q=problem[\'quotes\'][s];ks=__import__(\'surface_model\').QUOTES[s]\n        iv=[100*implied_vol(a,T,b) for a,b in zip(ks,q)]\n        axs[s].scatter(ks,iv,s=13,facecolor=\'white\',edgecolor=INK,zorder=5)\n        axs[s].set(title=f\'{T:g}-year implied volatility\',xlabel=\'Strike / forward\',ylabel=\'Volatility (%)\')\n        axs[s].grid(alpha=.16)\n    ymin=min(a.get_ylim()[0] for a in axs[:2]);ymax=max(a.get_ylim()[1] for a in axs[:2])\n    for a in axs[:2]:a.set_ylim(ymin,ymax)\n    categories=[(\'Exact\',r[r.method==\'exact\']),(\'FD\',r[r.method==\'fd\']),\n                (\'Matrix\\nLCHS\',r[r.method==\'lchs\'])]\n    for B in dict.fromkeys([low,high]):categories.append((f\'{B/1e6:.2f}M\\nshots\',quant[quant.shots==B]))\n    for i,(name,data) in enumerate(categories):\n        if len(data)==0:continue\n        vals=data.heldout_rmse_bp.to_numpy();col=BLUE if i>=3 else (ORANGE if i==2 else GREY)\n        axs[2].bar(i,vals.mean(),color=\'white\',edgecolor=col,width=.62,lw=1)\n        axs[2].scatter(i+np.linspace(-.13,.13,len(vals)),vals,s=18,color=col,zorder=4)\n    axs[2].set_xticks(range(len(categories)),[n for n,_ in categories],fontsize=6.5)\n    axs[2].set(ylabel=\'Held-out price RMSE (bp)\',title=\'All sampler seeds\',ylim=(0,None))\n    axs[2].grid(axis=\'y\',alpha=.16)\n    fig.legend(*axs[0].get_legend_handles_labels(),loc=\'lower center\',ncol=2,frameon=False,bbox_to_anchor=(.5,-.03))\n    fig.tight_layout(rect=(0,.18,1,1),w_pad=1.)\n    finish(fig,out,\'measured_calibration\',[\'new_experiment/calibration_results.csv\',\'new_experiment/configuration.json\'])\n    pd.DataFrame(records).to_csv(out/\'tables\'/\'measured_smile_data.csv\',index=False)\n    checks=pd.read_csv(run/\'circuit_validation.csv\')\n    def mean(B):\n        vals=quant[quant.shots==B].heldout_rmse_bp\n        return f\'{vals.mean():.2f}\' if len(vals) else \'not yet run\'\n    macros={\'LowShots\':f\'{low:,}\',\'HighShots\':f\'{high:,}\',\n        \'LowHeld\':mean(low),\'HighHeld\':mean(high),\'MatrixHeld\':f\'{mat.heldout_rmse_bp:.2f}\',\n        \'CircuitError\':r\'$\'+f\'{checks.absolute_error.max():.2e}\'.replace(\'e-\',r\'\\times10^{-\')+\'}$\',\n        \'RunProfile\':cfg[\'profile\'],\'MeasuredRuns\':str(len(quant))}\n    (out/\'tables\'/\'new_run_macros.tex\').write_text(\'\\n\'.join(\'\\\\newcommand{\\\\\'+k+\'}{\'+v+\'}\' for k,v in macros.items())+\'\\n\')\n    lines=[r\'\\begin{table}[h]\\centering\',r\'\\caption{New two-expiry circuit experiment. Held-out errors are in bp; ranges cover \'+str(len(cfg[\'seeds\']))+r\' sampler seeds. Shots/evaluation sum both maturities.}\\label{tab:measured}\',\n        r\'\\small\\setlength{\\tabcolsep}{4pt}\',r\'\\begin{tabular}{lrrr}\\toprule\',\n        r\'Objective & Mean RMSE & Range & Shots/evaluation\\\\\\midrule\']\n    for name,m in [(\'Exact\',\'exact\'),(\'Finite differences\',\'fd\'),(\'Matrix LCHS\',\'lchs\')]:\n        a=r[r.method==m].iloc[0];lines.append(f\'{name} & {a.heldout_rmse_bp:.2f} & --- & 0\'+r\'\\\\\')\n    for B in dict.fromkeys([low,high]):\n        d=quant[quant.shots==B]\n        if len(d):lines.append(f\'Measured LCHS & {d.heldout_rmse_bp.mean():.2f} & {d.heldout_rmse_bp.min():.2f}--{d.heldout_rmse_bp.max():.2f} & {2*B/1e6:.2f}M\'+r\'\\\\\')\n    lines += [r\'\\bottomrule\\end{tabular}\',r\'\\end{table}\']\n    (out/\'tables\'/\'measured_results.tex\').write_text(\'\\n\'.join(lines)+\'\\n\')\n    if (run/\'resource_sweep.csv\').exists():\n        d=pd.read_csv(run/\'resource_sweep.csv\');keys=sorted(set(zip(d.N,d.J)))\n        fig,axs=plt.subplots(1,2,figsize=(3.48,2.2),sharey=True)\n        for S,col,mark in [(1,ORANGE,\'o\'),(2,BLUE,\'s\')]:\n            a=d[d.expiries==S];y=np.array([keys.index((n,j)) for n,j in zip(a.N,a.J)])+(S-1.5)*.17\n            axs[0].scatter(a.qubits,y,c=col,marker=mark,s=17,label=f\'{S} interval\'+(\'s\' if S>1 else \'\'))\n            axs[1].scatter(a.cx,y,c=col,marker=mark,s=17)\n        axs[0].set_yticks(range(len(keys)),[f\'N={n}, J={j}\' for n,j in keys],fontsize=6.5)\n        axs[0].invert_yaxis();axs[0].set(xlabel=\'Qubits\',xlim=(0,None));axs[1].set(xlabel=\'CNOT gates\',xscale=\'log\')\n        for a in axs:a.grid(axis=\'x\',alpha=.18)\n        fig.legend(*axs[0].get_legend_handles_labels(),loc=\'lower center\',ncol=2,frameon=False,bbox_to_anchor=(.54,-.02))\n        fig.tight_layout(rect=(0,.10,1,1),w_pad=.8)\n        finish(fig,out,\'circuit_resources\',[\'new_experiment/resource_sweep.csv\'])\n    save_json(out/\'tables\'/\'new_result_summary.json\',dict(profile=cfg[\'profile\'],completed_runs=len(quant),\n        planned_runs=len(cfg[\'shots\'])*len(cfg[\'seeds\']),max_circuit_probability_error=float(checks.absolute_error.max()),\n        measured_low_mean_bp=mean(low),measured_high_mean_bp=mean(high),\n        target=\'same-grid synthetic exact PDE\',optimizer_uses=\'measurement counts only\'))\n    return r\n\ndef quadrature_diagnostic(out):\n    out=Path(out);rows=[]\n    for n in [2,3,4]:\n        p=data_for(grid(n))\n        for J in [8,16,32,64,128]:\n            for cutoff in [8.,12.,24.,32.,48.]:\n                m=financial_metrics(p,TRUTH,\'lchs\',J,cutoff)\n                rows.append(dict(N=2**n,J=J,cutoff=cutoff,**m))\n    d=pd.DataFrame(rows);d.to_csv(out/\'quadrature_diagnostic.csv\',index=False)\n    return d\n', 'legacy_figures.py': '"""Finance-facing figures from saved predictions, with explicit display interpolation.\nNo supplied Python module is executed and no calibration is repeated.\n"""\nfrom pathlib import Path\nimport argparse, json, io\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import ndtr\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm\nfrom matplotlib.lines import Line2D\n\nBLUE=\'#286F9F\'; ORANGE=\'#CB702C\'; INK=\'#252A2D\'; GREY=\'#737C82\'\nSTYLE={\'target\':(INK,\'-\',1.7),\'expm\':(GREY,\'--\',1.2),\'fd\':(GREY,\':\',1.2),\n       \'lchs-beta\':(BLUE,\'-\',1.35),\'lchs-simple\':(\'#78A9C8\',\'--\',1.2),\n       \'AH\':(ORANGE,\'--\',1.2),\'PINN-fixed\':(\'#D7A176\',\':\',1.3),\'PINN-Wam\':(\'#8F491C\',\'-.\',1.3)}\nLABEL={\'target\':\'Fine-grid target\',\'expm\':\'PDE: matrix exp.\',\'fd\':\'PDE: finite diff.\',\n       \'lchs-beta\':\'PDE: LCHS beta\',\'lchs-simple\':\'PDE: LCHS simple\',\'AH\':\'Andreasen-Huge\',\n       \'PINN-fixed\':\'PINN: fixed\',\'PINN-Wam\':\'PINN: adaptive\'}\nTIMES=np.array([182/365,1.]);XI_TRUTH=np.array([[.8,.3,.25,.7],[.6,.35,.3,.5]])\nMETHODS=[\'expm\',\'fd\',\'lchs-beta\',\'lchs-simple\',\'AH\',\'PINN-fixed\',\'PINN-Wam\']\nCASES=[\'in_bp0_d101\',\'out_bp0_d101\',\'out_bp5_d101\']\n\ndef read_json(p):return json.loads(Path(p).read_text())\ndef call(k,T,vol):\n    sd=vol*np.sqrt(T);d=-np.log(k)/sd+sd/2\n    return ndtr(d)-k*ndtr(d-sd)\n\ndef black_iv(price,k,T):\n    """Vectorised bracketed inversion; infeasible prices remain NaN, never clipped."""\n    p,k,t=np.broadcast_arrays(np.asarray(price,float),np.asarray(k,float),np.asarray(T,float))\n    valid=np.isfinite(p)&(t>0)&(p>np.maximum(1-k,0)+1e-12)&(p<1-1e-12)\n    valid&=(call(k,t,np.full(p.shape,1e-6))<=p)&(call(k,t,np.full(p.shape,8.))>=p)\n    low=np.full(p.shape,1e-6);high=np.full(p.shape,8.)\n    for _ in range(64):\n        mid=(low+high)/2;below=call(k,t,mid)<p\n        low=np.where(below,mid,low);high=np.where(below,high,mid)\n    result=np.where(valid,(low+high)/2,np.nan)\n    if valid.any():assert np.max(np.abs(call(k[valid],t[valid],result[valid])-p[valid]))<2e-12\n    return result\n\ndef payload(root,case,method,n=6,seed=0):\n    return read_json(root/\'checkpoints\'/f\'{case}_n{n}_{method}_s{seed}.json\')[\'result\']\n\ndef profile(root,case,method,s,n=6,seed=0):\n    """Use native predictions for AH/PINN, preserving the original benchmark."""\n    data=read_json(root/\'raw\'/f\'{case}.json\');q=data[\'quotes\'][s];test=data[\'tests\'][s]\n    k=np.r_[q[\'k\'],test[\'k\']];target=np.r_[q[\'clean\'],test[\'price\']]\n    train=np.r_[np.ones(len(q[\'k\']),bool),np.zeros(len(test[\'k\']),bool)]\n    observed=np.r_[q[\'mid\'],np.full(len(test[\'k\']),np.nan)]\n    if method==\'target\':prices=target.copy()\n    else:\n        res=payload(root,case,method,n,seed)\n        if \'quote_predictions\' in res:\n            prices=np.r_[res[\'quote_predictions\'][s],res[\'test_predictions\'][s]]\n        else:\n            K=np.exp(np.linspace(-1.2,1.2,2**n+2));prices=np.interp(k,K,res[\'C\'][s])\n    ix=np.argsort(k);assert np.all(np.diff(k[ix])>0)\n    return pd.DataFrame({\'dataset\':case,\'method\':method,\'initialisation\':seed,\'N\':2**n,\n        \'T\':q[\'T\'],\'k\':k[ix],\'is_training\':train[ix],\'observed_price\':observed[ix],\n        \'target_price\':target[ix],\'price\':prices[ix],\n        \'iv_pct\':100*black_iv(prices[ix],k[ix],q[\'T\']),\n        \'target_iv_pct\':100*black_iv(target[ix],k[ix],q[\'T\']),\n        \'price_error_bp\':1e4*(prices[ix]-target[ix])})\n\ndef basis(x):\n    z=(np.asarray(x)[None,:]-np.array([-.6,-.2,.2,.6])[:,None])/.45\n    return .04*(1+np.cos(np.pi*np.clip(z,-1,1)))*(np.abs(z)<=1)\n\ndef true_variance(k,s,truth):\n    x=np.log(k)\n    return (.01+XI_TRUTH[s]@basis(x)) if truth==\'in\' else (.01+.055*np.exp(-((x+.1)/.3)**2)+.035/(1+np.exp(4*x)))\n\ndef line(ax,x,y,method,label=None):\n    color,ls,lw=STYLE[method];ax.plot(x,y,color=color,ls=ls,lw=lw,label=label or LABEL[method])\n\ndef finish(fig,out,name,source,note):\n    for ax in fig.axes:\n        if ax.name!=\'3d\':ax.grid(alpha=.15,lw=.5)\n    for ext in [\'pdf\',\'png\']:\n        stream=io.BytesIO()\n        fig.savefig(stream,format=ext,dpi=240,bbox_inches=\'tight\',pad_inches=.04)\n        content=stream.getvalue()\n        if ext==\'pdf\':assert content.rstrip().endswith(b\'%%EOF\'), \'Incomplete PDF stream\'\n        destination=out/\'figures\'/f\'{name}.{ext}\'\n        temporary=destination.with_suffix(destination.suffix+\'.tmp\')\n        temporary.write_bytes(content);temporary.replace(destination)\n    (out/\'figures\'/f\'{name}.json\').write_text(json.dumps({\'source\':source,\'note\':note},indent=2))\n    plt.close(fig)\n\ndef generate(root,out):\n    root=Path(root);out=Path(out)\n    for d in [\'figures\',\'tables\',\'finance_data\']: (out/d).mkdir(parents=True,exist_ok=True)\n    plt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'font.size\':8,\'axes.titlesize\':8,\n        \'axes.labelsize\':8,\'legend.fontsize\':7,\'xtick.labelsize\':7,\'ytick.labelsize\':7,\n        \'axes.spines.top\':False,\'axes.spines.right\':False,\'pdf.fonttype\':42})\n    profiles=pd.concat([profile(root,c,m,s) for c in CASES for m in [\'target\',*METHODS] for s in range(2)],ignore_index=True)\n    recorded=pd.read_csv(root/\'results.csv\');reconciled=0\n    for c in CASES:\n        for m in METHODS:\n            d=profiles[(profiles.dataset==c)&(profiles.method==m)]\n            row=recorded[(recorded.dataset==c)&(recorded.method==m)&(recorded.n==6)&(recorded.seed==0)].iloc[0]\n            hold=d[~d.is_training];train=d[d.is_training]\n            assert len(hold)==int(row.iv_total)\n            assert abs(np.sqrt(np.mean(hold.price_error_bp**2))-row.rmse_heldout*1e4)<2e-9\n            assert abs(np.sqrt(np.mean((train.price-train.observed_price)**2))-row.rmse_quotes)<2e-12\n            reconciled+=1\n    profiles.to_csv(out/\'finance_data\'/\'saved_smiles.csv\',index=False)\n    def get(c,m,s):return profiles[(profiles.dataset==c)&(profiles.method==m)&np.isclose(profiles[\'T\'],TIMES[s])]\n    # Four surfaces: identical camera, scales and maturity interpolation rule.\n    case=CASES[-1];k=np.linspace(.7,1.4,73);tt=np.linspace(TIMES[0],TIMES[1],25)\n    K,TT=np.meshgrid(k,tt);surface_methods=[\'target\',\'lchs-beta\',\'AH\',\'PINN-Wam\']\n    interpolated={};records=[];surfaces={}\n    for m in surface_methods:\n        p=np.array([np.interp(k,get(case,m,s).k,get(case,m,s).price) for s in range(2)])\n        w=(tt-TIMES[0])/(TIMES[1]-TIMES[0]);C=(1-w[:,None])*p[0]+w[:,None]*p[1]\n        V=100*black_iv(C,K,TT);assert np.isfinite(V).all();surfaces[m]=V;interpolated[m]=p\n        records.append(pd.DataFrame({\'dataset\':case,\'method\':m,\'N\':64,\'initialisation\':0,\n            \'T_display\':TT.ravel(),\'k\':K.ravel(),\'price_display\':C.ravel(),\'iv_display_pct\':V.ravel(),\n            \'display_rule\':\'Linear interpolation of saved prices between two expiries\'}))\n    pd.concat(records,ignore_index=True).to_csv(out/\'finance_data\'/\'surface_display.csv\',index=False)\n    lo=np.floor(min(v.min() for v in surfaces.values()));hi=np.ceil(max(v.max() for v in surfaces.values()))\n    fig=plt.figure(figsize=(7.16,4.0));gs=fig.add_gridspec(2,4,height_ratios=[1.3,1.],hspace=.30,wspace=.08,\n        top=.90,bottom=.10,left=.11,right=.92)\n    for i,m in enumerate(surface_methods):\n        ax=fig.add_subplot(gs[0,i],projection=\'3d\')\n        ax.plot_surface(K,TT,surfaces[m],color=STYLE[m][0],alpha=.78,edgecolor=\'white\',linewidth=.12,\n            rstride=2,cstride=3,shade=False,antialiased=True)\n        ax.view_init(elev=25,azim=-125);ax.set_box_aspect((1.1,.8,.85))\n        ax.set(xlim=(.7,1.4),ylim=(TIMES[0],1),zlim=(lo,hi))\n        ax.set_xticks([.7,1.,1.4]);ax.set_yticks([.5,1.]);ax.set_zticks([lo,(lo+hi)/2,hi])\n        ax.set_xlabel(\'K / F\',labelpad=-5);ax.set_ylabel(\'Years\',labelpad=-6)\n        ax.set_zlabel(\'IV (%)\' if i==0 else \'\',labelpad=-1);ax.tick_params(pad=-2,labelsize=6)\n        ax.set_title(f\'({chr(97+i)}) \'+LABEL[m],pad=1,fontweight=\'bold\',fontsize=7)\n        for axis in [ax.xaxis,ax.yaxis,ax.zaxis]:axis.pane.fill=False;axis._axinfo[\'grid\'][\'color\']=(.8,.8,.8,.35)\n    ax=fig.add_subplot(gs[1,:]);errors=np.vstack([1e4*(interpolated[m]-interpolated[\'target\']) for m in surface_methods[1:]])\n    cmap=LinearSegmentedColormap.from_list(\'signed\',[BLUE,\'#FAFAFA\',ORANGE]);lim=float(np.ceil(np.abs(errors).max()))\n    mesh=ax.imshow(errors,origin=\'upper\',extent=[.7,1.4,6,0],aspect=\'auto\',cmap=cmap,vmin=-lim,vmax=lim,\n        interpolation=\'nearest\')\n    ax.set_yticks(np.arange(6)+.5,[LABEL[m].replace(\'PDE: \',\'\')+f\' / {"6m" if s==0 else "1y"}\' for m in surface_methods[1:] for s in range(2)])\n    ax.set_xticks([.7,.85,1.,1.2,1.4]);ax.set_xlabel(\'Normalised strike K / F\')\n    ax.set_title(\'(e) Price errors at the two fitted expiries\',loc=\'left\',fontweight=\'bold\',pad=4)\n    for yy in [2,4]:ax.axhline(yy,color=\'white\',lw=1.5)\n    cb=fig.colorbar(mesh,ax=ax,pad=.015,fraction=.018);cb.set_label(\'Model minus target (bp)\',fontsize=7);cb.ax.tick_params(labelsize=7)\n    fig.suptitle(\'Out-of-family surface / noise +/-5 bp / N=64 / dataset 101, initialisation 0\',fontsize=9,y=.995)\n    finish(fig,out,\'surface_comparison\',[\'finance_data/saved_smiles.csv\',\'finance_data/surface_display.csv\'],\n        \'3D maturity values are linear price interpolation for display, not additional calibration or PDE solutions. The error heatmap uses the two fitted expiries only.\')\n    # Shared-scale smiles and signed price errors, across three named cases.\n    fig,axs=plt.subplots(2,3,figsize=(7.16,3.35),sharex=True,sharey=\'row\',layout=\'constrained\',\n        gridspec_kw={\'height_ratios\':[1.25,1]})\n    titles=[\'In-family / clean\',\'Out-of-family / clean\',\'Out-of-family / noise +/-5 bp\']\n    allvol=[];allerr=[]\n    for j,c in enumerate(CASES):\n        for m in [\'target\',\'lchs-beta\',\'AH\',\'PINN-Wam\']:\n            d=get(c,m,1);line(axs[0,j],d.k,d.iv_pct,m);allvol.extend(d.iv_pct)\n            if m!=\'target\':line(axs[1,j],d.k,d.price_error_bp,m);allerr.extend(d.price_error_bp)\n        q=read_json(root/\'raw\'/f\'{c}.json\')[\'quotes\'][1]\n        obs=100*black_iv(q[\'mid\'],q[\'k\'],q[\'T\'])\n        axs[0,j].scatter(q[\'k\'],obs,s=21,facecolors=\'none\',edgecolors=INK,lw=.85,zorder=5,label=\'Input quotes\')\n        axs[0,j].set_title(f\'({chr(97+j)}) {titles[j]}\',fontweight=\'bold\',fontsize=8)\n        axs[1,j].axhline(0,color=INK,lw=.75);axs[1,j].set_xlabel(\'Normalised strike K / F\')\n        axs[1,j].set_xlim(.7,1.4);axs[1,j].set_xticks([.7,1.,1.4])\n    ylim=(np.floor(np.nanmin(allvol))-1,np.ceil(np.nanmax(allvol))+1)\n    for ax in axs[0]:ax.set_ylim(*ylim)\n    lim=np.ceil(np.max(np.abs(allerr)));axs[1,0].set_ylim(-lim,lim)\n    axs[0,0].set_ylabel(\'1-year implied volatility (%)\');axs[1,0].set_ylabel(\'Price error (bp)\')\n    handles,labels=axs[0,0].get_legend_handles_labels();fig.legend(handles,labels,ncol=5,loc=\'outside lower center\',frameon=False,fontsize=7)\n    fig.suptitle(\'Smile shape and pricing error / N=64 / dataset 101, initialisation 0\',fontsize=9)\n    finish(fig,out,\'smile_comparison\',[\'finance_data/saved_smiles.csv\'],\n        \'Fixed examples at maturity one year. Native AH/PINN predictions are retained. Input quotes are noisy observations in the third column. Lower panels share one signed bp scale.\')\n    # Local volatility is distinct from implied volatility. Only stored PDE coefficients are used.\n    fig,axs=plt.subplots(2,2,figsize=(3.5,2.65),sharex=True,sharey=True,layout=\'constrained\');records=[]\n    k=np.linspace(.7,1.4,201)\n    for j,truth in enumerate([\'in\',\'out\']):\n        c=f\'{truth}_bp0_d101\'\n        for s in range(2):\n            target=100*np.sqrt(true_variance(k,s,truth));line(axs[s,j],k,target,\'target\',label=\'True local vol.\')\n            for m in [\'expm\',\'lchs-beta\']:\n                hats=np.asarray(payload(root,c,m)[\'hats\']);vol=100*np.sqrt(.01+hats[s]@basis(np.log(k)))\n                line(axs[s,j],k,vol,m,label=\'Exact PDE\' if m==\'expm\' else \'LCHS beta\')\n                if m==\'expm\':axs[s,j].plot(k[::30],vol[::30],ls=\'none\',marker=\'o\',ms=2.4,mfc=\'white\',mec=GREY,mew=.6,zorder=6)\n                records.append(pd.DataFrame({\'dataset\':c,\'method\':m,\'slice\':s+1,\'k\':k,\'local_vol_pct\':vol,\'target_local_vol_pct\':target}))\n            axs[s,j].set_title((\'In-family\' if j==0 else \'Out-of-family\')+(\' / 0-6m\' if s==0 else \' / 6-12m\'),fontsize=7)\n            axs[s,j].set_xlim(.7,1.4);axs[s,j].set_xticks([.7,1.,1.4]);axs[s,j].set_ylim(10,32)\n            if j==0:axs[s,j].set_ylabel(\'Local vol. (%)\')\n            if s==1:axs[s,j].set_xlabel(\'K / F\')\n    handles,labels=axs[0,0].get_legend_handles_labels();fig.legend(handles,labels,ncol=3,loc=\'outside lower center\',frameon=False,fontsize=6.5)\n    pd.concat(records).to_csv(out/\'finance_data\'/\'local_volatility.csv\',index=False)\n    finish(fig,out,\'local_volatility_recovery\',[\'finance_data/local_volatility.csv\'],\n        \'Clean data, N=64, initialisation0. Local volatility is sqrt(local variance); piecewise constant in time. AH/PINN coefficient predictions were not saved and are not imputed.\')\n    # Financial cost of a nonnegative-payoff butterfly, at identical fitted coefficients.\n    key=\'out_bp0_d101_n7_lchs-beta_s0\';r=read_json(root/\'checkpoints\'/f\'{key}.json\')[\'result\']\n    K=np.exp(np.linspace(-1.2,1.2,130));C=np.asarray(r[\'C\']);E=np.asarray(r[\'C_exact\'])\n    dm=np.diff(K)[:-1];dp=np.diff(K)[1:];bf=lambda z:(dm*z[:,2:]+dp*z[:,:-2])/(dm+dp)-z[:,1:-1]\n    s=0;raw=bf(C)[s]*1e4;exact=bf(E)[s]*1e4;band=(K[:-2]>=.7)&(K[2:]<=1.4)\n    minimum=np.where(band,raw,np.inf).argmin();assert raw[minimum]<0\n    fig,axs=plt.subplots(2,1,figsize=(3.5,2.75),sharex=True,layout=\'constrained\')\n    line(axs[0],K,E[s]*100,\'expm\',label=\'Exact, same coefficients\');line(axs[0],K,C[s]*100,\'lchs-beta\',label=\'Raw LCHS beta\')\n    axs[0].set_ylabel(\'Call price (% of D F)\');axs[0].set_title(\'Clean out-of-family / N=128 / 6 months / start 0\',fontsize=7.5)\n    axs[0].set_ylim(0,105*max(np.interp(.7,K,C[s]),np.interp(.7,K,E[s])))\n    axs[0].legend(frameon=False,fontsize=6.5,loc=\'upper right\')\n    line(axs[1],K[1:-1],exact,\'expm\');line(axs[1],K[1:-1],raw,\'lchs-beta\')\n    axs[1].axhline(0,color=INK,lw=.8);axs[1].fill_between(K[1:-1],raw,0,where=raw<0,color=BLUE,alpha=.2)\n    axs[1].scatter(K[minimum+1],raw[minimum],marker=\'v\',s=24,color=BLUE,zorder=5)\n    axs[1].annotate(f\'{raw[minimum]:.2f} bp\',xy=(K[minimum+1],raw[minimum]),xytext=(1.13,-2.3),fontsize=7,\n        arrowprops={\'arrowstyle\':\'->\',\'lw\':.7,\'color\':INK})\n    axs[1].set(xlim=(.7,1.4),xlabel=\'Normalised strike K / F\',ylabel=\'Butterfly cost (bp)\')\n    axs[1].set_xticks([.7,.85,1.,1.2,1.4]);axs[1].set_ylim(min(-3.5,raw[band].min()-.3),max(5,raw[band].max()+.3))\n    pd.DataFrame({\'source_key\':key,\'T\':TIMES[s],\'k_left\':K[:-2],\'k_centre\':K[1:-1],\'k_right\':K[2:],\n        \'raw_butterfly_bp\':raw,\'exact_butterfly_bp\':exact,\'inside_quote_band\':band}).to_csv(out/\'finance_data\'/\'butterfly_portfolios.csv\',index=False)\n    finish(fig,out,\'arbitrage_finance\',[f\'checkpoints/{key}.json\',\'finance_data/butterfly_portfolios.csv\'],\n        \'Upper: prices at identical fitted coefficients. Lower: cost of long weighted wing calls and short centre call; negative cost violates the nonnegative butterfly payoff. Displayed points use the original strike grid.\')\n    # Numerical comparison with financially interpretable units and dataset replication.\n    summ=pd.read_csv(out/\'audit\'/\'summary_verified.csv\');res=pd.read_csv(root/\'results.csv\')\n    rows=[r\'\\begin{table*}[t]\\centering\\caption{Held-out call-price RMSE (normalised price bp). Noisy entries are means $\\pm$ sample SD over three datasets after averaging five starts; AH is deterministic. Times are CPU calibration seconds per run in the out-of-family noisy case, excluding compilation and evaluation.}\\label{tab:performance}\',\n          r\'\\small\\setlength{\\tabcolsep}{5pt}\\begin{tabular}{lrrrrr}\\toprule & \\multicolumn{2}{c}{In-family target} & \\multicolumn{2}{c}{Out-of-family target} & CPU time\\\\\',\n          r\'Method & Clean & Noise $\\pm5$ bp & Clean & Noise $\\pm5$ bp & Seconds\\\\\\midrule\']\n    ivrecords=[]\n    for m in METHODS:\n        values=[]\n        for truth,noise in [(\'in\',0),(\'in\',5),(\'out\',0),(\'out\',5)]:\n            r=summ[(summ.n==6)&(summ.method==m)&(summ.truth==truth)&(summ.noise_bp==noise)].iloc[0]\n            v=f\'{r.heldout*1e4:.2f}\'\n            if noise:v=f\'${v}\\\\pm{r.heldout_sd*1e4:.2f}$\'\n            if m==\'AH\' and truth==\'in\' and noise:v+=r\'$^{\\dagger}$\'\n            values.append(v)\n        r=summ[(summ.n==6)&(summ.method==m)&(summ.truth==\'out\')&(summ.noise_bp==5)].iloc[0]\n        rows.append(LABEL[m].replace(\'Andreasen-Huge\',\'Andreasen--Huge\')+\' & \'+\' & \'.join(values)+f\' & {r.opt_s:.3g}\'+r\'\\\\\')\n        rr=res[(res.n==6)&(res.method==m)&(res.truth==\'out\')&(res.noise_bp==5)]\n        ivrecords.append({\'method\':m,\'heldout_price_bp\':r.heldout*1e4,\'opt_seconds\':r.opt_s,\n                         \'iv_valid\':int(rr.iv_valid.sum()),\'iv_total\':int(rr.iv_total.sum())})\n    rows.extend([r\'\\bottomrule\\end{tabular}\',r\'\\par\\vspace{2pt}\\footnotesize $^{\\dagger}$AH uses two feasible noisy in-family datasets; one input quote is below intrinsic value. A paired two-dataset comparison is provided in the supplement.\\end{table*}\'])\n    (out/\'tables\'/\'financial_performance.tex\').write_text(\'\\n\'.join(rows))\n    pd.DataFrame(ivrecords).to_csv(out/\'finance_data\'/\'price_time_iv_coverage.csv\',index=False)\n    fig,axs=plt.subplots(1,2,figsize=(6.5,2.25),layout=\'constrained\',sharey=True)\n    for i,m in enumerate(METHODS[:4]):\n        r=summ[(summ.n==6)&(summ.method==m)&(summ.truth==\'out\')&(summ.noise_bp==5)].iloc[0]\n        axs[0].errorbar(r.heldout*1e4,i,xerr=r.heldout_sd*1e4,fmt=\'o\',color=STYLE[m][0],capsize=2)\n        axs[1].scatter(r.opt_s,i,color=STYLE[m][0],marker=[\'o\',\'s\',\'D\',\'^\'][i],s=25)\n    axs[0].set_yticks(range(4),[LABEL[m] for m in METHODS[:4]]);axs[0].invert_yaxis();axs[0].set_xlim(left=0)\n    axs[0].set_xlabel(\'Held-out price RMSE (bp)\');axs[1].set_xscale(\'log\');axs[1].set_xlabel(\'CPU calibration time (seconds)\')\n    axs[0].set_title(\'Same model: price accuracy\');axs[1].set_title(\'Same model: classical computation\')\n    finish(fig,out,\'solver_comparison\',[\'audit/summary_verified.csv\'],\n        \'N64, out-of-family, noise5bp, three independent datasets and five starts. Time excludes compilation and evaluation; no quantum runtime is measured.\')\n    checks={\'native_prediction_source_preserved\':True,\'iv_inversion_repricing_tolerance\':2e-12,\n        \'illustrated_run_metrics_reconciled\':reconciled,\n        \'surface_interpolation\':\'linear prices between saved expiries; display only\',\'new_maturities_calibrated\':0,\n        \'surface_case\':case,\'smile_cases\':CASES,\'example_start\':0,\'band_butterfly_example_min_bp\':float(raw[band].min()),\n        \'all_surface_display_iv_finite\':True,\'plots\':[\'surface_comparison\',\'smile_comparison\',\'local_volatility_recovery\',\'arbitrage_finance\',\'solver_comparison\']}\n    (out/\'finance_data\'/\'figure_checks.json\').write_text(json.dumps(checks,indent=2))\n    print(json.dumps(checks,indent=2));return checks\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--input\',required=True);p.add_argument(\'--out\',required=True)\n    a=p.parse_args();generate(a.input,a.out)\n', 'quantum_experiments.py': '"""Shot-based, multi-expiry LCHS calibration with an explicit circuit boundary.\n\nOnly counts returned by ``runner.run`` enter the measured optimiser. Dense\nsmall-system Hamiltonian unitaries are compiled, and their classical synthesis\ncost is recorded. This is an executable prototype, not an efficient-oracle claim.\n"""\nimport os, sys, time, hashlib, platform, importlib.metadata\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy.linalg import expm, svd\nfrom scipy.stats import beta as beta_distribution\nfrom qiskit import QuantumCircuit, transpile, qpy\nfrom qiskit.circuit.library import StatePreparation, UnitaryGate\nfrom qiskit.quantum_info import Statevector\nfrom qiskit_aer import AerSimulator\nfrom surface_model import (grid, data_for, generator, weights, prices, loss,\n                           financial_metrics, save_json, TIMES, TRUTH)\n\ndef dilation(B):\n    u,s,vh=svd(B); z=np.sqrt(np.maximum(0,1-s*s)); v=vh.conj().T\n    R=np.block([[B,(u*z)@u.conj().T],[(v*z)@v.conj().T,-B.conj().T]])\n    if not np.allclose(R.conj().T@R,np.eye(len(R)),atol=1e-11):\n        raise AssertionError(\'Nonunitary observation dilation\')\n    return R\n\ndef build_residual(problem, xi, expiry, terms=32, cutoff=32.):\n    """Residual through expiry inclusive, retaining distinct LCU workspaces.\n\n    Input coefficients are classical. Later expiries evolve from the original\n    payoff through EVERY preceding interval, without reading intermediate states.\n    """\n    g=problem[\'g\']; N=g[\'N\']; n=g[\'n\']; S=expiry+1\n    b=int(np.log2(terms)); assert 2**b==terms\n    xi=np.asarray(xi).reshape(len(problem[\'times\']),g[\'M\'])\n    k,w,meta=weights(terms,cutoff); alpha=meta[\'alpha\']\n    work=list(range(n)); kernels=[list(range(n+s*b,n+(s+1)*b)) for s in range(S)]\n    wa=n+S*b; branch=wa+1; width=branch+1\n    obs=problem[\'observations\'][expiry]; W0=obs[:,1:-1]; Q=len(obs)\n    if Q>N: raise ValueError(\'Small prototype requires number of quotes <= grid size\')\n    W=np.zeros((N,N)); W[:Q]=W0\n    y=np.zeros(N); y[:Q]=problem[\'quotes\'][expiry]-obs@g[\'G\']\n    gamma=max(1.,np.linalg.norm(W,2)); unorm=np.linalg.norm(g[\'v0\']); ynorm=np.linalg.norm(y)\n    T=problem[\'times\'][expiry]\n    a=unorm*np.exp(g[\'shift\']*T)*alpha**S*gamma; scale=a+ynorm\n    prep=StatePreparation(np.sqrt(abs(w)/alpha))\n    forward=QuantumCircuit(width-1,name=f\'PDE-T{expiry+1}\')\n    forward.append(StatePreparation(g[\'v0\']/unorm),work)\n    last=0.\n    for s in range(S):\n        dt=problem[\'times\'][s]-last; last=problem[\'times\'][s]\n        A=generator(g,xi[s]); H=-.5j*(A-A.T); L=(A+A.T)/2+g[\'shift\']*np.eye(N)\n        kr=kernels[s]; forward.append(prep,kr)\n        for j in range(terms):\n            U=np.exp(1j*np.angle(w[j]))*expm(-1j*dt*(H+k[j]*L))\n            # There is no enumeration of coefficient designs or preloaded loss.\n            gate=UnitaryGate(U,label=f\'U{s},{j}\').control(b,ctrl_state=j)\n            forward.append(gate,kr+work)\n        forward.append(prep.inverse(),kr)\n    forward.append(UnitaryGate(dilation(W/gamma),label=\'Quote map\'),work+[wa])\n    V=QuantumCircuit(width,name=f\'Residual-T{expiry+1}\')\n    mix=StatePreparation([np.sqrt(a/scale),np.sqrt(ynorm/scale)])\n    V.append(mix,[branch]); V.x(branch)\n    V.append(forward.to_gate().control(1),[branch]+list(range(width-1)))\n    V.x(branch)\n    V.append(StatePreparation(y/ynorm).control(1),[branch]+work)\n    V.z(branch); V.append(mix.inverse(),[branch])\n    anc=[q for kr in kernels for q in kr]+[wa,branch]\n    return V,dict(expiry=expiry,quotes=Q,scale=scale,scale_to_mse=scale**2/Q,\n                  ancillas=anc,qubits=width,terms=terms,cutoff=cutoff,alpha=alpha,\n                  price_amplification=float(np.exp(g[\'shift\']*T)*alpha**S))\n\ndef compile_residual(problem,xi,expiry,terms,cutoff):\n    start=time.perf_counter(); V,meta=build_residual(problem,xi,expiry,terms,cutoff)\n    V.add_register(__import__(\'qiskit\').ClassicalRegister(len(meta[\'ancillas\']),\'success\'))\n    V.measure(meta[\'ancillas\'],range(len(meta[\'ancillas\'])))\n    qc=transpile(V,basis_gates=[\'u\',\'cx\'],optimization_level=1,seed_transpiler=17)\n    meta.update(cx=int(qc.count_ops().get(\'cx\',0)),u=int(qc.count_ops().get(\'u\',0)),\n                depth=qc.depth(),compile_seconds=time.perf_counter()-start)\n    return qc,meta\n\nclass MeasuredObjective:\n    """Inject an AerSimulator or a compatible Qiskit backend runner.\n\n    A real backend requires target-aware transpilation and suitable logical\n    resources. No job is sent to any external backend by the default notebook.\n    """\n    def __init__(self,problem,shots=262144,seed=101,terms=32,cutoff=32.,runner=None,cache=None):\n        self.problem=problem; self.shots=shots; self.seed=seed\n        self.terms=terms; self.cutoff=cutoff\n        self.runner=runner or AerSimulator(method=\'statevector\',max_parallel_threads=1)\n        self.cache={} if cache is None else cache\n        self.records=[]; self.call=0\n    def __call__(self,xi):\n        self.call+=1; circuits=[]; metadata=[]\n        xi=np.asarray(xi).reshape(len(self.problem[\'times\']),self.problem[\'g\'][\'M\'])\n        for s in range(len(self.problem[\'times\'])):\n            # Cache only circuits for an identical parameter vector, never losses.\n            key=(s,tuple(xi[:s+1].ravel()),self.terms,self.cutoff)\n            if key not in self.cache:\n                self.cache[key]=compile_residual(self.problem,xi,s,self.terms,self.cutoff)\n            qc,meta=self.cache[key]; circuits.append(qc); metadata.append(meta)\n        started=time.perf_counter()\n        run_options={\'shots\':self.shots}\n        if isinstance(self.runner,AerSimulator):\n            run_options[\'seed_simulator\']=self.seed+1009*self.call\n        result=self.runner.run(circuits,**run_options).result()\n        elapsed=time.perf_counter()-started; estimates=[]; variances=[]\n        for s,meta in enumerate(metadata):\n            counts=result.get_counts(s); hits=counts.get(\'0\'*len(meta[\'ancillas\']),0)\n            p=hits/self.shots; fac=meta[\'scale_to_mse\']\n            tail=.05/(2*len(metadata))\n            lower=0. if hits==0 else beta_distribution.ppf(tail,hits,self.shots-hits+1)\n            upper=1. if hits==self.shots else beta_distribution.ppf(1-tail,hits+1,self.shots-hits)\n            estimates.append(fac*p); variances.append(fac**2*p*(1-p)/self.shots)\n            self.records.append(dict(call=self.call,expiry=s+1,shots=self.shots,hits=hits,\n                probability=p,mse_estimate=fac*p,mse_standard_error=np.sqrt(variances[-1]),\n                mse_lower=fac*lower,mse_upper=fac*upper,\n                sampler_seed=self.seed,parameters=xi.ravel().tolist(),\n                cx_per_execution=meta[\'cx\'],depth=meta[\'depth\'],qubits=meta[\'qubits\'],\n                backend_seconds=elapsed/len(metadata),compile_seconds=meta[\'compile_seconds\']))\n        return float(np.mean(estimates))\n\ndef coordinate_search(objective,shape=(2,2),rounds=2,step=.25):\n    """Identical bounded pattern search for measured and matrix objectives.\n\n    The incumbent is remeasured for every search direction. All candidates have the\n    same shot budget. No exact prices, gradients or validation scores are used.\n    """\n    xi=np.full(shape,.5); history=[]; evaluation=0\n    directions=list(np.eye(xi.size))\n    # Opposing within-expiry changes directly explore smile tilt as well as level.\n    for s in range(shape[0]):\n        d=np.zeros(xi.size);d[s*shape[1]]=1;d[s*shape[1]+1]=-1;directions.append(d)\n    for sweep in range(rounds):\n        for j,direction in enumerate(directions):\n            trials=[]\n            candidates=[xi.copy()]\n            for sign in [-1,1]:\n                x=np.clip(xi.ravel()+sign*step*direction,0,1).reshape(shape)\n                if not any(np.array_equal(x,y) for y in candidates):candidates.append(x)\n            for x in candidates:\n                f=objective(x);evaluation+=1\n                trials.append((f,x));history.append(dict(sweep=sweep,coordinate=j,evaluation=evaluation,\n                    loss=f,parameters=x.ravel().tolist()))\n            _,xi=min(trials,key=lambda z:z[0])\n    return xi,pd.DataFrame(history)\n\ndef circuit_checks(problem,terms=8,cutoff=4.,out=None):\n    rows=[]\n    for xi in [np.full((2,2),.5),TRUTH]:\n        p=prices(problem[\'g\'],xi,method=\'lchs\',terms=terms,cutoff=cutoff)\n        for s in range(2):\n            V,meta=build_residual(problem,xi,s,terms,cutoff)\n            state=Statevector.from_instruction(V).data\n            # Work qubits are low bits, all successful ancillary bits are zero.\n            observed=float(np.sum(abs(state[:problem[\'g\'][\'N\']])**2))\n            expected=float(np.mean(abs(problem[\'observations\'][s]@p[s]-problem[\'quotes\'][s])**2)/meta[\'scale_to_mse\'])\n            error=abs(observed-expected)\n            if error>2e-10:raise AssertionError(f\'Multi-expiry residual mismatch: {error}\')\n            rows.append(dict(expiry=s+1,parameters=xi.ravel().tolist(),\n                             probability=observed,expected=expected,absolute_error=error))\n    frame=pd.DataFrame(rows)\n    if out:frame.to_csv(Path(out)/\'circuit_validation.csv\',index=False)\n    return frame\n\ndef run_calibration_experiment(out,profile=\'paper\',runner=None):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    configs={\'smoke\':dict(terms=4,cutoff=4.,shots=[4096],seeds=[101],rounds=1),\n             \'pilot\':dict(terms=8,cutoff=12.,shots=[65536,262144],seeds=[101,202],rounds=1),\n             \'paper\':dict(terms=8,cutoff=12.,shots=[262144,4194304],seeds=[101,202,303],rounds=2),\n             \'precision\':dict(terms=32,cutoff=32.,shots=[1048576,16777216],seeds=[101,202,303],rounds=2)}\n    cfg=configs[profile];problem=data_for(grid());cache={}\n    save_json(out/\'configuration.json\',dict(version=\'2026-09-09-r2\',profile=profile,**cfg,\n       target=\'same-grid synthetic exact PDE\',truth=TRUTH,quotes=problem[\'quotes\'],\n       grid=problem[\'g\'][\'K\'],times=TIMES,noise_bp=0,hardware_execution=False,\n       implementation=\'coherent LCHS within each maturity; measured residual; classical bounded pattern search\',\n       synthesis=\'dense small-system Hamiltonian unitaries; all-to-all u/cx\',\n       optimiser_access=\'measurement counts only; no statevector, exact objective or gradient\'))\n    save_json(out/\'environment.json\',dict(python=platform.python_version(),\n        packages={p:importlib.metadata.version(p) for p in [\'numpy\',\'scipy\',\'pandas\',\'qiskit\',\'qiskit-aer\']},\n        source_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest()\n                       for p in [Path(__file__),Path(__file__).with_name(\'surface_model.py\')]},\n        sampling=\'Aer finite-shot counts from decomposed circuits\',\n        seed_policy=\'base sampler seed + 1009 * objective call index\'))\n    circuit_checks(problem,min(cfg[\'terms\'],8),min(cfg[\'cutoff\'],12),out)\n    results=[];histories=[];measurements=[]\n    for method in [\'exact\',\'fd\',\'lchs\']:\n        start=time.perf_counter()\n        x,h=coordinate_search(lambda z:loss(problem,z,method,cfg[\'terms\'],cfg[\'cutoff\']),rounds=cfg[\'rounds\'])\n        elapsed=time.perf_counter()-start\n        row=dict(method=method,shots=0,seed=0,parameters=x.ravel().tolist(),evaluations=len(h),cpu_seconds=elapsed,\n                 **financial_metrics(problem,x,method,cfg[\'terms\'],cfg[\'cutoff\']))\n        results.append(row);h[\'method\']=method;h[\'shots\']=0;h[\'seed\']=0;histories.append(h)\n    for shots in cfg[\'shots\']:\n        for seed in cfg[\'seeds\']:\n            print(f\'Measured calibration: shots={shots}, seed={seed}\',flush=True)\n            obj=MeasuredObjective(problem,shots,seed,cfg[\'terms\'],cfg[\'cutoff\'],runner,cache)\n            start=time.perf_counter();x,h=coordinate_search(obj,rounds=cfg[\'rounds\']);elapsed=time.perf_counter()-start\n            row=dict(method=\'quantum_counts\',shots=shots,seed=seed,parameters=x.ravel().tolist(),evaluations=len(h),\n                     cpu_seconds=elapsed,**financial_metrics(problem,x,\'lchs\',cfg[\'terms\'],cfg[\'cutoff\']))\n            row[\'total_shots\']=sum(r[\'shots\'] for r in obj.records)\n            row[\'total_cx_executions\']=sum(r[\'shots\']*r[\'cx_per_execution\'] for r in obj.records)\n            row[\'max_qubits\']=max(r[\'qubits\'] for r in obj.records)\n            results.append(row);h[\'method\']=\'quantum_counts\';h[\'shots\']=shots;h[\'seed\']=seed;histories.append(h)\n            # Independent final evaluation is not used to select the coefficients.\n            final_start=len(obj.records);row[\'independent_final_measured_loss\']=obj(x)\n            final_records=obj.records[final_start:]\n            row[\'independent_final_mse_lower\']=float(np.mean([r[\'mse_lower\'] for r in final_records]))\n            row[\'independent_final_mse_upper\']=float(np.mean([r[\'mse_upper\'] for r in final_records]))\n            for record in obj.records:\n                record[\'purpose\']=\'calibration\' if record[\'call\']<=len(h) else \'independent_final_check\'\n            measurements.extend(obj.records)\n            row[\'verification_shots\']=len(final_records)*shots\n            row[\'verification_cx_executions\']=sum(r[\'shots\']*r[\'cx_per_execution\'] for r in final_records)\n            # Incremental checkpoints retain completed runs if Colab disconnects.\n            pd.DataFrame(results).to_csv(out/\'calibration_results.csv\',index=False)\n            pd.concat(histories,ignore_index=True).to_csv(out/\'optimisation_history.csv\',index=False)\n            pd.DataFrame(measurements).to_json(out/\'measurement_log.json\',orient=\'records\',indent=2)\n            save_json(out/\'checkpoint.json\',dict(completed=len(results)-3,planned=len(cfg[\'shots\'])*len(cfg[\'seeds\']),last=row))\n    # Export actual compiled circuits for the final measured coefficients.\n    circuits=[];resource=[]\n    for s in range(2):\n        qc,meta=compile_residual(problem,x,s,cfg[\'terms\'],cfg[\'cutoff\'])\n        circuits.append(qc);resource.append(meta)\n    with (out/\'final_residual_circuits.qpy\').open(\'wb\') as f:qpy.dump(circuits,f)\n    save_json(out/\'final_circuit_resources.json\',resource)\n    save_json(out/\'status.json\',dict(completed=True,profile=profile,hardware_execution=False,\n        measured_runs=len(cfg[\'shots\'])*len(cfg[\'seeds\']),no_statevector_access_in_optimiser=True))\n    return pd.DataFrame(results)\n\ndef resource_sweep(out,profile=\'paper\'):\n    out=Path(out); rows=[]\n    cases=[(2,4,1),(2,4,2),(2,8,1),(2,8,2),(3,4,1),(3,4,2)]\n    if profile==\'paper\':cases += [(2,16,1),(2,16,2),(2,32,1),(2,32,2),(3,8,1),(3,8,2)]\n    for n,J,S in cases:\n        p=data_for(grid(n));x=np.full((2,2),.5)\n        cutoff=4. if J<=8 else 32.\n        _,m=compile_residual(p,x,S-1,J,cutoff)\n        rows.append(dict(N=2**n,J=J,cutoff=cutoff,expiries=S,**{k:m[k] for k in [\'qubits\',\'cx\',\'u\',\'depth\',\'compile_seconds\',\'price_amplification\']}))\n        print(f\'Resource circuit N={2**n}, J={J}, expiries={S}: {m["qubits"]} qubits\',flush=True)\n    df=pd.DataFrame(rows);df.to_csv(out/\'resource_sweep.csv\',index=False);return df\n', 'submission_figures.py': '"""Figures for the revised manuscript: architecture, surfaces and measured precision."""\nfrom pathlib import Path\nimport json,io\nimport numpy as np,pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import FancyBboxPatch,FancyArrowPatch\nfrom legacy_figures import black_iv\nfrom surface_model import save_json\n\nBLUE=\'#25618C\';GOLD=\'#A36B25\';INK=\'#202124\';GREY=\'#777777\'\ndef style():\n    plt.rcParams.update({\'font.family\':\'STIXGeneral\',\'mathtext.fontset\':\'stix\',\'font.size\':9,\n        \'axes.titlesize\':10,\'axes.labelsize\':9,\'legend.fontsize\':8,\'xtick.labelsize\':8,\n        \'ytick.labelsize\':8,\'axes.spines.top\':False,\'axes.spines.right\':False,\'pdf.fonttype\':42})\ndef save(fig,out,name,sources):\n    out=Path(out);(out/\'figures\').mkdir(exist_ok=True,parents=True)\n    for ext in [\'pdf\',\'png\']:\n        stream=io.BytesIO();fig.savefig(stream,format=ext,dpi=240,bbox_inches=\'tight\',pad_inches=.03)\n        (out/\'figures\'/f\'{name}.{ext}\').write_bytes(stream.getvalue())\n    save_json(out/\'figures\'/f\'{name}.json\',dict(sources=sources))\n    plt.close(fig)\n\ndef architecture(out):\n    """Original maturity-indexed diagram of this paper\'s actual hybrid objective."""\n    from matplotlib.patches import Rectangle, Circle\n    style();fig,ax=plt.subplots(figsize=(7.15,2.8))\n    ax.set(xlim=(0,14.3),ylim=(0,5.6));ax.axis(\'off\')\n    edge=\'#30434b\';gate=\'#e6f0f1\';classical=\'#f1ece3\'\n    def box(x,y,w,h,label,fill=\'white\',size=9):\n        ax.add_patch(Rectangle((x,y),w,h,facecolor=fill,edgecolor=edge,lw=.7,zorder=3))\n        ax.text(x+w/2,y+h/2,label,ha=\'center\',va=\'center\',fontsize=size,color=INK,zorder=4)\n    def arr(points,ls=\'-\',colour=edge):\n        if len(points)>2:\n            xx,yy=zip(*points[:-1]);ax.plot(xx,yy,color=colour,lw=.8,ls=ls)\n        ax.add_patch(FancyArrowPatch(points[-2],points[-1],arrowstyle=\'->\',mutation_scale=8,\n                                     linewidth=.8,color=colour,linestyle=ls))\n    ax.text(3.25,5.32,r\'Candidate local-variance coefficients $\\xi=(\\xi_1,\\ldots,\\xi_S)$\',ha=\'center\',fontsize=9)\n    ax.plot([.75,5.85],[5.05,5.05],color=edge,lw=.6)\n    ax.text(7.65,5.32,\'Quote discrepancy\',ha=\'center\',fontsize=9)\n    ax.text(10.25,5.32,\'Circuit counts\',ha=\'center\',fontsize=9)\n    xs=[1.65,2.85,4.0,5.1]\n    for row,(lab,y,indices) in enumerate([(\'1\',4.32,[0]),(\'2\',3.02,[0,1]),(\'S\',1.72,[0,1,2,3])]):\n        ax.text(.62,y+.38,r\'$T_\'+lab+\'$\',ha=\'center\',fontsize=9)\n        ax.text(.62,y-.02,r\'$v_0$\',ha=\'center\',fontsize=10)\n        ax.plot([.87,6.2],[y,y],color=edge,lw=.8)\n        for ind in indices:\n            x=xs[ind]\n            if ind==2:\n                ax.text(x,y,r\'$\\cdots$\',ha=\'center\',va=\'center\',fontsize=13,\n                        bbox=dict(facecolor=\'white\',edgecolor=\'none\',pad=.5))\n            else:\n                j=[\'1\',\'2\',\'\',\'S\'][ind]\n                box(x-.4,y-.29,.8,.58,r\'$\\widetilde P_\'+j+\'$\',gate,11)\n        box(6.2,y-.34,2.85,.68,r\'$W_\'+lab+r\'\\widetilde v_\'+lab+r\'-y_\'+lab+\'$\',size=10)\n        arr([(9.08,y),(9.7,y)])\n        ax.add_patch(Circle((10.05,y),.19,edgecolor=edge,facecolor=\'white\',lw=.7))\n        # A measurement dial, kept distinct from the unitary pricing blocks.\n        theta=np.linspace(.2,np.pi-.2,35)\n        ax.plot(10.05+.125*np.cos(theta),y-.04+.125*np.sin(theta),color=edge,lw=.6)\n        ax.plot([10.05,10.135],[y-.075,y+.07],color=edge,lw=.7)\n        ax.text(10.67,y,r\'$h_\'+lab+r\'/B_\'+lab+\'$\',ha=\'center\',va=\'center\',fontsize=10)\n        ax.plot([11.18,11.55],[y,y],color=edge,lw=.75)\n    ax.plot([11.55,11.55],[1.72,4.32],color=edge,lw=.75)\n    arr([(11.55,3.02),(12.02,3.02)])\n    box(12.05,2.40,1.95,1.24,\'Scale and\\naverage\\n\'+r\'$\\widehat F(\\xi)$\',classical,9)\n    ax.text(3.35,.98,\'Each maturity restarts from the payoff\',ha=\'center\',fontsize=8)\n    box(7.65,.20,3.4,.65,r\'Classical optimiser: update $\\xi$\',classical,9)\n    arr([(13.03,2.36),(13.03,.525),(11.10,.525)])\n    arr([(7.60,.525),(.12,.525),(.12,5.32),(.65,5.32)])\n    ax.text(3.5,.32,\'next candidate\',ha=\'center\',va=\'top\',fontsize=8,color=edge)\n    fig.subplots_adjust(left=.006,right=.994,top=.99,bottom=.005)\n    save(fig,out,\'calibration_architecture\',[\'surface_model.py: residual circuit construction\',\n         \'quantum_experiments.py: count-driven optimisation\',\'main.tex: propagation and residual equations\'])\n\n\ndef surfaces(examples,out):\n    """Column-width financial figure: each row combines a surface and a slice."""\n    style();examples=Path(examples);out=Path(out)\n    d=pd.read_csv(examples/\'surface_predictions.csv\')\n    quotes={c:json.loads((examples/f\'{c}_quotes.json\').read_text()) for c in [\'smile\',\'skew\']}\n    d[\'iv_pct\']=100*black_iv(d.price,d.k,d[\'T\'])\n    d[\'target_iv_pct\']=100*black_iv(d.target_price,d.k,d[\'T\'])\n    assert np.isfinite(d.iv_pct).all(), \'Invalid IV in the displayed strike/maturity window\'\n    d.to_csv(examples/\'surface_plot_data.csv\',index=False)\n    fig=plt.figure(figsize=(3.5,3.05))\n    for j,case in enumerate([\'smile\',\'skew\']):\n        y=.58 if j==0 else .11\n        ax=fig.add_axes([-.035,y, .67,.385],projection=\'3d\')\n        z=d[(d.case==case)&(d.method==\'lchs\')].pivot(index=\'T\',columns=\'k\',values=\'iv_pct\')\n        K,T=np.meshgrid(z.columns,z.index)\n        ax.plot_surface(K,T,z.to_numpy(),color=BLUE,alpha=.66,edgecolor=\'none\',\n                        antialiased=True,rstride=1,cstride=4)\n        q=quotes[case];qk,qt=np.meshgrid(q[\'k\'],q[\'times\']);qiv=100*black_iv(q[\'prices\'],qk,qt)\n        ax.scatter(qk.ravel(),qt.ravel(),qiv.ravel(),c=INK,s=3,depthshade=False)\n        ax.set(xlabel=r\'$k$\',ylabel=r\'$T$\',zlabel=\'IV (%)\')\n        ax.set_xticks([.7,1,1.4]);ax.set_yticks([.25,1]);ax.set_zticks([20,22] if j==0 else [20,24])\n        ax.view_init(elev=23,azim=-55);ax.set_proj_type(\'ortho\')\n        ax.set_box_aspect((1.55,1,.9),zoom=1.12)\n        ax.tick_params(pad=-1,labelsize=6.5)\n        ax.xaxis.labelpad=-8;ax.yaxis.labelpad=-9;ax.zaxis.labelpad=-5\n        ax.xaxis.label.set_size(8);ax.yaxis.label.set_size(8);ax.zaxis.label.set_size(7)\n        for axis in [ax.xaxis,ax.yaxis,ax.zaxis]:\n            axis.pane.fill=False;axis._axinfo[\'grid\'][\'linewidth\']=.35\n        fig.text(.045,y+.39,f\'({chr(97+j)}) \'+case.title(),fontsize=9)\n        ax=fig.add_axes([.62025,y+.075,.36975,.27])\n        a=d[(d.case==case)&(d[\'T\']==1.)];t=a[a.method==\'exact\'];b=a[a.method==\'lchs\']\n        ax.plot(t.k,t.target_iv_pct,color=INK,lw=1.1,label=\'Target\')\n        ax.plot(t.k,t.iv_pct,color=GREY,lw=1.1,ls=\'--\',label=\'Exact\')\n        ax.plot(b.k,b.iv_pct,color=BLUE,lw=1.05,ls=\':\',label=\'LCHS\')\n        qiv=100*black_iv(np.array(q[\'prices\'])[-1],np.array(q[\'k\']),1.)\n        ax.scatter(q[\'k\'],qiv,facecolor=\'white\',edgecolor=INK,s=7,lw=.5,label=\'Quotes\',zorder=5)\n        ax.set(title=r\'$T=1$\',xlabel=r\'$k$\',xlim=(.7,1.4))\n        ax.set_xticks([.7,1,1.4]);ax.set_yticks([21.6,22.2] if j==0 else [21,24])\n        ax.tick_params(labelsize=6.5,pad=1,length=2)\n        ax.xaxis.labelpad=0;ax.xaxis.label.set_size(8);ax.title.set_size(8)\n        ax.grid(alpha=.16,lw=.5)\n    h,l=ax.get_legend_handles_labels()\n    fig.legend(h,l,loc=\'lower center\',ncol=4,frameon=False,bbox_to_anchor=(.5,.005),\n               fontsize=7,handlelength=1.5,columnspacing=.75,handletextpad=.35)\n    save(fig,out,\'financial_surfaces\',[\'surface_examples/surface_predictions.csv\',\'surface_examples/*_quotes.json\'])\n\ndef precision(run,out):\n    style();run=Path(run);d=pd.read_csv(run/\'calibration_results.csv\');d=d[d.method==\'quantum_counts\'].copy()\n    fig,ax=plt.subplots(figsize=(3.45,1.9))\n    B=sorted(d.shots.unique());labels=[]\n    for i,(_,r) in enumerate(d.sort_values([\'shots\',\'seed\']).iterrows()):\n        est=1e4*np.sqrt(r.independent_final_measured_loss)\n        low=1e4*np.sqrt(r.independent_final_mse_lower);high=1e4*np.sqrt(r.independent_final_mse_upper)\n        colour=BLUE if r.shots==B[0] else GOLD\n        ax.errorbar(est,i,xerr=np.array([[est-low],[high-est]]),fmt=\'o\',color=colour,ms=3,capsize=2,lw=.8)\n        ax.scatter(r.quote_rmse_bp,i,marker=\'x\',color=INK,s=19,lw=1,zorder=4)\n        labels.append(f\'{r.shots/1e6:.2f}M / {int(r.seed)}\')\n    ax.set_yticks(range(len(labels)),labels,fontsize=7);ax.invert_yaxis()\n    ax.set(xlabel=\'Final quote RMSE (price bp)\',ylabel=\'Shots / seed\',xlim=(0,None))\n    ax.grid(axis=\'x\',alpha=.18,lw=.5)\n    from matplotlib.lines import Line2D\n    handles=[Line2D([],[],color=INK,marker=\'x\',ls=\'\',label=\'Matrix value\'),Line2D([],[],color=GREY,marker=\'o\',ls=\'-\',label=\'Measurement + 95% CI\')]\n    ax.legend(handles=handles,loc=\'lower right\',fontsize=7,frameon=False)\n    fig.tight_layout(pad=.3)\n    save(fig,out,\'measurement_precision\',[\'new_experiment/calibration_results.csv\'])\n\ndef generate(examples,run,out):\n    architecture(out);surfaces(examples,out);precision(run,out);numerical_tables(run,out)\n    a=pd.read_csv(Path(examples)/\'surface_accuracy.csv\')\n    vals={}\n    for case in [\'smile\',\'skew\']:\n        for m,tag in [(\'exact\',\'Exact\'),(\'lchs\',\'Lchs\')]:\n            vals[case.title()+tag]=f\'{a[(a.case==case)&(a.method==m)].iloc[0].heldout_rmse_bp:.2f}\'\n    out=Path(out);(out/\'tables\').mkdir(exist_ok=True)\n    (out/\'tables/surface_macros.tex\').write_text(\'\\n\'.join(\'\\\\newcommand{\\\\\'+k+\'}{\'+v+\'}\' for k,v in vals.items())+\'\\n\')\n    return vals\n\n\ndef numerical_tables(run,out):\n    run=Path(run);out=Path(out);(out/\'tables\').mkdir(exist_ok=True,parents=True)\n    cfg=json.loads((run/\'configuration.json\').read_text())\n    r=pd.read_csv(run/\'calibration_results.csv\');q=r[r.method==\'quantum_counts\']\n    lo,hi=cfg[\'shots\'][0],cfg[\'shots\'][-1]\n    a=q[q.shots==lo];b=q[q.shots==hi];m=r[r.method==\'lchs\'].iloc[0]\n    checks=pd.read_csv(run/\'circuit_validation.csv\')\n    mant,exp=f\'{checks.absolute_error.max():.2e}\'.split(\'e\')\n    vals={\'LowShots\':f\'{lo:,}\',\'HighShots\':f\'{hi:,}\',\n        \'LowHeld\':f\'{a.heldout_rmse_bp.mean():.2f}\',\'HighHeld\':f\'{b.heldout_rmse_bp.mean():.2f}\',\n        \'MatrixHeld\':f\'{m.heldout_rmse_bp:.2f}\',\n        \'LowCoeff\':f\'{a.coefficient_rmse.mean():.3f}\',\'HighCoeff\':f\'{b.coefficient_rmse.mean():.3f}\',\n        \'CircuitError\':\'$\'+mant+r\'\\times10^{\'+str(int(exp))+\'}$\',\n        \'RunProfile\':cfg[\'profile\'],\'MeasuredRuns\':str(len(q))}\n    (out/\'tables/new_run_macros.tex\').write_text(\'\\n\'.join(\'\\\\newcommand{\\\\\'+k+\'}{\'+v+\'}\' for k,v in vals.items())+\'\\n\')\n    rows=[]\n    for method,label in [(\'exact\',\'Exact\'),(\'fd\',\'Finite differences\'),(\'lchs\',\'Matrix LCHS\')]:\n        z=r[r.method==method].iloc[0]\n        rows.append(f\'{label} & n/a & {z.heldout_rmse_bp:.2f} & {z.coefficient_rmse:.3f}\'+r\'\\\\\')\n    for shots in dict.fromkeys([lo,hi]):\n        z=q[q.shots==shots]\n        rows.append(f\'Measured LCHS & $2^{{{int(np.log2(shots))}}}$ & {z.heldout_rmse_bp.mean():.2f} & {z.coefficient_rmse.mean():.3f}\'+r\'\\\\\')\n    (out/\'tables/measured_rows.tex\').write_text(\'\\n\'.join(rows)+r\'\\bottomrule\'+\'\\n\')\n    d=pd.read_csv(run/\'resource_sweep.csv\');rows=[]\n    for (N,J),z in d.groupby([\'N\',\'J\']):\n        one=z[z.expiries==1].iloc[0];two=z[z.expiries==2].iloc[0]\n        rows.append(f\'{N} & {J} & {int(one.qubits)} & {int(one.cx):,} & {int(two.qubits)} & {int(two.cx):,}\'+r\'\\\\\')\n    (out/\'tables/resource_rows.tex\').write_text(\'\\n\'.join(rows)+r\'\\bottomrule\'+\'\\n\')\n    return vals\n\n\ndef consistency(example,out):\n    """Accuracy and butterfly comparison at one fixed coefficient vector."""\n    style();example=Path(example);out=Path(out)\n    d=pd.read_csv(example/\'consistency_predictions.csv\')\n    meta=json.loads((example/\'consistency_validation.json\').read_text())\n    fig,axes=plt.subplots(2,1,figsize=(3.45,2.50),sharex=True)\n    specs=[(\'exact\',\'Exact\',INK,\'--\',1.25),\n           (\'fd\',\'FD (1,024 steps)\',GREY,\':\',1.05),\n           (\'lchs_2048\',r\'LCHS $J=2{,}048$\',GOLD,\'-\',1.05),\n           (\'lchs_4096\',r\'LCHS $J=4{,}096$\',BLUE,\'-\',1.15)]\n    for method,label,col,ls,lw in specs:\n        a=d[(d.method==method)&d.in_quote_band]\n        axes[0].plot(a.k,a.error_bp,label=label,color=col,ls=ls,lw=lw,\n                     marker=\'o\' if method==\'fd\' else None,ms=2,markevery=7,mfc=\'white\')\n        b=d[(d.method==method)&d.complete_butterfly_in_band]\n        axes[1].plot(b.k,b.butterfly_bp,label=label,color=col,ls=ls,lw=lw,\n                     marker=\'o\' if method==\'fd\' else None,ms=2,markevery=7,mfc=\'white\')\n    for ax in axes:\n        ax.set(xlim=(.7,1.4));ax.set_xticks([.7,.9,1.1,1.3,1.4])\n        ax.tick_params(labelsize=7,pad=1,length=2)\n        ax.grid(alpha=.16,lw=.5);ax.axhline(0,color=GREY,lw=.5,zorder=0)\n        ax.xaxis.labelpad=0;ax.yaxis.labelpad=2\n    axes[0].set(ylabel=\'Error (bp)\',ylim=(-4,4.7))\n    axes[0].text(.015,.95,\'(a) Price error\',ha=\'left\',va=\'top\',transform=axes[0].transAxes,fontsize=8)\n    axes[0].set_yticks([-4,0,4])\n    axes[0].text(.98,.98,r\'$N=128$, $T=182/365$\',ha=\'right\',va=\'top\',transform=axes[0].transAxes,fontsize=8)\n    axes[1].set(ylabel=\'Butterfly (bp)\',xlabel=r\'$k$\',ylim=(-4,10))\n    axes[1].text(.015,.95,\'(b) Butterfly price\',ha=\'left\',va=\'top\',transform=axes[1].transAxes,fontsize=8)\n    axes[1].set_yticks([-4,0,4,8]);axes[1].axhspan(-4,0,facecolor=\'#f4e8e3\',edgecolor=\'none\',zorder=-1)\n    axes[1].text(1.38,-3.35,\'Negative\',ha=\'right\',color=\'#8f422c\',fontsize=8)\n    h,l=axes[1].get_legend_handles_labels()\n    fig.legend(h,l,ncol=2,loc=\'lower center\',bbox_to_anchor=(.55,-.01),frameon=False,\n               fontsize=7,handlelength=2.0,columnspacing=1.2,labelspacing=.3)\n    fig.subplots_adjust(left=.135,right=.988,top=.985,bottom=.275,hspace=.15)\n    save(fig,out,\'financial_consistency\',[\'consistency_example/consistency_predictions.csv\',\n         \'consistency_example/consistency_validation.json\',\'paper/evidence/checkpoints/\'+meta[\'checkpoint\']+\'.json\'])\n    # Values inserted into the manuscript derive from the plotted arrays.\n    a=meta[\'metrics\'];vals={\'CoarseMax\':f"{a[\'lchs_2048\'][\'max_inband_price_error_bp\']:.2f}",\n        \'RefinedMax\':f"{a[\'lchs_4096\'][\'max_inband_price_error_bp\']:.3f}",\n        \'FDMax\':f"{a[\'fd\'][\'max_inband_price_error_bp\']:.3f}",\n        \'CoarseButterfly\':f"{a[\'lchs_2048\'][\'min_inband_butterfly_bp\']:.2f}"}\n    (out/\'tables\').mkdir(exist_ok=True,parents=True)\n    (out/\'tables/consistency_macros.tex\').write_text(\'\\n\'.join(\'\\\\newcommand{\\\\\'+k+\'}{\'+v+\'}\' for k,v in vals.items())+\'\\n\')\n    return meta\n', 'surface_example.py': '"""Synthetic smile/skew examples in a positive quadratic LV family.\n\nCalibrate with classical exact propagation, then compare exact, backward Euler\nand finite LCHS at the SAME fitted coefficients. This experiment isolates the\nsurface representation and forward approximation; it is not a measured-circuit\nrun or an independently optimised LCHS calibration.\n"""\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy.linalg import eigh_tridiagonal,eigvalsh\nfrom scipy.sparse import diags\nfrom scipy.sparse.linalg import expm_multiply\nfrom scipy.optimize import least_squares\nfrom surface_model import observation,weights,save_json\n\nTIMES=np.array([.25,.5,.75,1.])\nQUOTES=np.array([.70,.78,.87,.95,1.,1.06,1.15,1.28,1.4])\nCOEFFICIENTS={\n \'smile\':np.array([[.10,.50,.50],[.16,.50,.50],[.22,.50,.50],[.28,.50,.50]]),\n \'skew\':np.array([[.20,.10,.85],[.22,.12,.80],[.24,.14,.75],[.26,.16,.70]])}\n\ndef make_grid(N):\n    x=np.linspace(-1.2,1.2,N+2);k=np.exp(x);h=x[1]-x[0]\n    drift=2*np.tanh(h/2)/h\n    Q=(np.full(N-1,-1/h**2-drift/(2*h)),np.full(N,2/h**2),np.full(N-1,-1/h**2+drift/(2*h)))\n    xi=x[1:-1]\n    phi=np.array([np.full(N,.08),.1*(xi+.45)**2,.1*(xi-.45)**2])\n    G=(1-k[0])*(k[-1]-k)/(k[-1]-k[0]);v0=np.maximum(1-k[1:-1],0)-G[1:-1]\n    return dict(N=N,K=k,x=x,h=h,Q=Q,phi=phi,G=G,v0=v0)\n\ndef entries(g,xi):\n    a=.01+np.asarray(xi)@g[\'phi\'];lo,d,up=g[\'Q\']\n    return .5*a[1:]*lo,.5*a*d,.5*a[:-1]*up\n\ndef advance(g,xi,dt,v,method=\'exact\',J=2048,cutoff=48.):\n    lo,d,up=entries(g,xi)\n    if method in [\'exact\',\'fd\']:\n        log_r=np.r_[0.,np.cumsum(.5*(np.log(-lo)-np.log(-up)))];log_r-=np.mean(log_r);r=np.exp(log_r)\n        ev,Q=eigh_tridiagonal(d,-np.sqrt(lo*up))\n        factor=np.exp(-dt*ev) if method==\'exact\' else (1+dt*ev/1024)**-1024\n        return r*(Q@(factor*(Q.T@(v/r))))\n    # Complex Hermitian tridiagonals are reduced by a diagonal unitary gauge.\n    shift=g[\'shift\'];ks,ws,_=weights(J,cutoff)\n    result=np.zeros(g[\'N\'],complex)\n    for k,w in zip(ks,ws):\n        b=k*(up+lo)/2+.5j*(lo-up)\n        phase=np.r_[1.+0j,np.exp(-1j*np.cumsum(np.angle(b)))]\n        ev,Q=eigh_tridiagonal(k*(d+shift),abs(b))\n        result+=w*phase*(Q@(np.exp(-1j*dt*ev)*(Q.T@(phase.conj()*v))))\n    return np.exp(shift*dt)*result\n\ndef add_shift(g):\n    vals=[np.full(g[\'N\'],.01),*g[\'phi\']]\n    lo,d,up=g[\'Q\'];shift=0.\n    for a in vals:\n        diag=.5*a*d;off=.25*(a[:-1]*up+a[1:]*lo)\n        smallest=eigh_tridiagonal(diag,off,select=\'i\',select_range=(0,0),eigvals_only=True)[0]\n        shift+=max(0.,-smallest)\n    g[\'shift\']=shift;return g\n\ndef surface(g,coefficients,method=\'exact\',times=TIMES,J=2048,cutoff=48.):\n    v=g[\'v0\'].astype(complex);last=0.;out=[]\n    for s,T in enumerate(times):\n        v=advance(g,coefficients[s],T-last,v,method,J,cutoff);last=T\n        c=g[\'G\'].astype(complex).copy();c[1:-1]+=v;out.append(c)\n    return np.array(out)\n\ndef run(out,profile=\'paper\'):\n    import time\n    out=Path(out);out.mkdir(exist_ok=True,parents=True)\n    N=128 if profile!=\'smoke\' else 64;refN=1024 if profile!=\'smoke\' else 256\n    J=8192 if profile!=\'smoke\' else 4096;cutoff=96.\n    target_grid=make_grid(refN);g=add_shift(make_grid(N));O=observation(g,QUOTES)\n    ks=np.linspace(.7,1.4,141);W=observation(g,ks);Wref=observation(target_grid,ks)\n    rows=[];fits=[];timings=[]\n    for case,truth in COEFFICIENTS.items():\n        target=surface(target_grid,truth).real\n        quotes=target@observation(target_grid,QUOTES).T\n        v=g[\'v0\'].astype(complex);last=0.;coeff=[];t0=time.perf_counter()\n        for s,T in enumerate(TIMES):\n            def residual(x):\n                c=g[\'G\'].copy();c[1:-1]+=advance(g,x,T-last,v).real\n                return (O@c-quotes[s])*1e4\n            fit=least_squares(residual,np.full(g[\'phi\'].shape[0],.5),bounds=(0,1),ftol=1e-10,xtol=1e-10,gtol=1e-9,max_nfev=100)\n            coeff.append(fit.x);v=advance(g,fit.x,T-last,v);last=T\n            fits.append(dict(case=case,T=T,parameters=fit.x.tolist(),success=bool(fit.success),nfev=fit.nfev,cost=fit.cost))\n        coeff=np.array(coeff);calib=time.perf_counter()-t0\n        for method in [\'exact\',\'fd\',\'lchs\']:\n            t0=time.perf_counter();p=surface(g,coeff,method,J=J,cutoff=cutoff);seconds=time.perf_counter()-t0\n            for s,T in enumerate(TIMES):\n                reference=Wref@target[s];pred=W@p[s]\n                rows.extend(dict(case=case,method=method,N=N,J=J,cutoff=cutoff,T=T,k=k,\n                    target_price=float(t),price=float(z.real),imaginary_price=float(z.imag),\n                    fit_scope=\'exact-calibrated common coefficients\') for k,t,z in zip(ks,reference,pred))\n            error=(p@W.T-target@Wref.T);quoted=(p@O.T-quotes)\n            held=~np.any(np.isclose(ks[:,None],QUOTES[None,:],atol=1e-10),axis=1)\n            timings.append(dict(case=case,method=method,N=N,J=J,cutoff=cutoff,\n                calibration_seconds=calib,forward_seconds=seconds,\n                heldout_rmse_bp=float(np.sqrt(np.mean(abs(error[:,held])**2))*1e4),\n                quote_rmse_bp=float(np.sqrt(np.mean(abs(quoted)**2))*1e4)))\n        print(\'Surface example complete:\',case,flush=True)\n        save_json(out/f\'{case}_quotes.json\',dict(k=QUOTES,times=TIMES,prices=quotes,truth=truth,coefficients=coeff))\n    pd.DataFrame(rows).to_csv(out/\'surface_predictions.csv\',index=False)\n    pd.DataFrame(timings).to_csv(out/\'surface_accuracy.csv\',index=False)\n    save_json(out/\'calibration_records.json\',fits)\n    save_json(out/\'configuration.json\',dict(profile=profile,N=N,reference_N=refN,J=J,cutoff=cutoff,times=TIMES,\n        examples=\'synthetic smile and skew from positive quadratic local variance; no market data\',\n        basis=[\'0.08\',\'0.1*(x+0.45)^2\',\'0.1*(x-0.45)^2\'],\n        calibration=\'classical exact propagation, sequential bounded least squares, common coefficients\',\n        methods=[\'exact\',\'fd\',\'lchs\'],shift=g[\'shift\']))\n    return pd.DataFrame(timings)\n\n\ndef validate_reference(out,profile=\'paper\'):\n    """Independent dense check and refinement of synthetic reference prices."""\n    from scipy.linalg import expm\n    out=Path(out)\n    g=add_shift(make_grid(16));xi=np.array([.2,.4,.7]);lo,d,up=entries(g,xi)\n    A=np.diag(d)+np.diag(lo,-1)+np.diag(up,1);v=g[\'v0\'].astype(complex);dt=.25\n    exact_error=float(np.max(abs(advance(g,xi,dt,v)-expm(-dt*A)@v)))\n    shift=g[\'shift\'];L=(A+A.T)/2+shift*np.eye(16);H=(A-A.T)/(2j)\n    nodes,ws,_=weights(8,12.)\n    dense=sum(w*expm(-1j*dt*(H+k*L))@v for k,w in zip(nodes,ws))*np.exp(shift*dt)\n    lchs_error=float(np.max(abs(advance(g,xi,dt,v,\'lchs\',8,12.)-dense)))\n    assert exact_error<1e-10 and lchs_error<1e-10\n    n=1024 if profile==\'paper\' else 256;g1=make_grid(n);g2=make_grid(2*n)\n    W1=observation(g1,QUOTES);W2=observation(g2,QUOTES)\n    refinement={}\n    for case,truth in COEFFICIENTS.items():\n        p1=surface(g1,truth)@W1.T;p2=surface(g2,truth)@W2.T\n        refinement[case]=dict(rmse_bp=float(np.sqrt(np.mean(abs(p1-p2)**2))*1e4),\n                             max_error_bp=float(np.max(abs(p1-p2))*1e4))\n    preds=pd.read_csv(out/\'surface_predictions.csv\')\n    result=dict(profile=profile,exact_dense_max_error=exact_error,lchs_dense_max_error=lchs_error,\n                reference_nodes=[n,2*n],reference_refinement=refinement,\n                max_imaginary_price=float(preds.imaginary_price.abs().max()),\n                all_fits_converged=all(z[\'success\'] for z in __import__(\'json\').loads((out/\'calibration_records.json\').read_text())))\n    assert result[\'all_fits_converged\'] and result[\'max_imaginary_price\']<1e-10\n    save_json(out/\'validation.json\',result)\n    return result\n', 'surface_model.py': '"""Financial model shared by classical references and measured quantum experiments.\n\nThe small circuit experiment deliberately uses same-grid synthetic quotes to\nisolate implementation error. It is not a fine-grid accuracy benchmark.\n"""\nfrom pathlib import Path\nimport json, hashlib, time, platform, importlib.metadata\nimport numpy as np\nimport pandas as pd\nfrom scipy.linalg import expm, eigvalsh, eigh\nfrom scipy.optimize import brentq\nfrom scipy.special import ndtr\n\nVERSION = \'2026-09-09-quantum-surface-r1\'\nTIMES = np.array([0.5, 1.0])\nQUOTES = [np.array([0.80, 0.95, 1.10, 1.30]),\n          np.array([0.85, 1.00, 1.15, 1.35])]\nTRUTH = np.array([[0.75, 0.25], [0.25, 0.75]])\n\ndef save_json(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    def convert(x):\n        if isinstance(x, np.ndarray): return x.tolist()\n        if isinstance(x, np.generic): return x.item()\n        if isinstance(x, Path): return str(x)\n        raise TypeError(type(x).__name__)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(value, default=convert, indent=2, allow_nan=False))\n    tmp.replace(path)\n\ndef grid(n=2, M=2, extent=1.2):\n    N = 2**n\n    x = np.linspace(-extent, extent, N+2); K = np.exp(x)\n    h = x[1]-x[0]; fitted = 2*np.tanh(h/2)/h\n    L = np.diag(np.full(N, 2/h**2))\n    L += np.diag(np.full(N-1, -1/h**2+fitted/(2*h)), 1)\n    L += np.diag(np.full(N-1, -1/h**2-fitted/(2*h)), -1)\n    centres = np.linspace(-0.55, 0.55, M)\n    width = 0.85 if M == 2 else 1.8/max(1,M-1)\n    z = (x[None,1:-1]-centres[:,None])/width\n    phi = 0.04*(1+np.cos(np.pi*np.clip(z,-1,1)))*(abs(z)<=1)\n    slope = (1-K[0])/(K[0]-K[-1]); G = slope*(K-K[-1])\n    A0 = .005*L; Am = .5*phi[:,:,None]*L[None,:,:]\n    shift = sum(max(0., -eigvalsh((a+a.T)/2)[0]) for a in [A0,*Am])\n    return dict(n=n,N=N,M=M,K=K,x=x,h=h,L=L,phi=phi,G=G,\n                A0=A0,Am=Am,shift=shift,\n                v0=np.maximum(1-K[1:-1],0)-G[1:-1])\n\ndef generator(g, xi): return g[\'A0\']+np.einsum(\'m,mij->ij\',xi,g[\'Am\'])\n\ndef observation(g, strikes):\n    K=g[\'K\']; strikes=np.asarray(strikes)\n    if np.any((strikes<K[0])|(strikes>K[-1])): raise ValueError(\'Quote outside domain\')\n    j=np.clip(np.searchsorted(K,strikes)-1,0,len(K)-2)\n    t=(strikes-K[j])/(K[j+1]-K[j])\n    W=np.zeros((len(strikes),len(K)))\n    W[np.arange(len(j)),j]=1-t; W[np.arange(len(j)),j+1]=t\n    return W\n\ndef weights(terms=8, cutoff=4., beta=.8):\n    k=-cutoff+(np.arange(terms)+.5)*(2*cutoff/terms)\n    w=np.exp(2**beta-(1+1j*k)**beta)/(2*np.pi*(1-1j*k))*(2*cutoff/terms)\n    defect=abs(w.sum()-1); w=w/w.sum()\n    return k,w,dict(alpha=float(abs(w).sum()),normalisation_defect=float(defect))\n\ndef propagator(g, xi, dt, method=\'exact\', terms=8, cutoff=4., fd_steps=128):\n    A=generator(g,xi)\n    if method==\'exact\': return expm(-dt*A)\n    if method==\'fd\': return np.linalg.matrix_power(np.linalg.inv(np.eye(g[\'N\'])+dt*A/fd_steps),fd_steps)\n    k,w,_=weights(terms,cutoff)\n    H=-.5j*(A-A.T); L=(A+A.T)/2+g[\'shift\']*np.eye(g[\'N\'])\n    vals,vecs=np.linalg.eigh(H[None,:,:]+k[:,None,None]*L[None,:,:])\n    U=np.einsum(\'kij,kj,klj->kil\',vecs,np.exp(-1j*dt*vals),vecs.conj())\n    return np.exp(dt*g[\'shift\'])*np.einsum(\'k,kij->ij\',w,U)\n\ndef prices(g, coefficients, times=TIMES, method=\'exact\', terms=8, cutoff=4.):\n    coefficients=np.asarray(coefficients).reshape(len(times),g[\'M\'])\n    v=g[\'v0\'].astype(complex); last=0.; result=[]\n    for xi,T in zip(coefficients,times):\n        v=propagator(g,xi,T-last,method,terms,cutoff)@v; last=T\n        c=g[\'G\'].astype(complex).copy(); c[1:-1]+=v\n        result.append(c)\n    return np.asarray(result)\n\ndef data_for(g, truth=TRUTH, noise_bp=0., seed=101):\n    p=prices(g,truth); rng=np.random.default_rng(seed)\n    obs=[observation(g,k) for k in QUOTES]\n    quotes=[(W@p[i]).real+rng.normal(0,noise_bp/1e4,len(W)) for i,W in enumerate(obs)]\n    return dict(g=g,quotes=quotes,observations=obs,truth=np.asarray(truth),\n                noise_bp=noise_bp,data_seed=seed,times=TIMES.copy())\n\ndef loss(problem, xi, method=\'lchs\', terms=8, cutoff=4.):\n    p=prices(problem[\'g\'],xi,problem[\'times\'],method,terms,cutoff)\n    return float(np.mean([np.mean(abs(W@p[s]-problem[\'quotes\'][s])**2)\n                          for s,W in enumerate(problem[\'observations\'])]))\n\ndef financial_metrics(problem, xi, method=\'exact\', terms=8, cutoff=4.):\n    g=problem[\'g\']; p=prices(g,xi,problem[\'times\'],method,terms,cutoff)\n    target=prices(g,problem[\'truth\'],problem[\'times\'])\n    strikes=np.linspace(.7,1.4,81); W=observation(g,strikes)\n    error=(p-target)@W.T\n    quoted=np.concatenate([O@p[s]-problem[\'quotes\'][s] for s,O in enumerate(problem[\'observations\'])])\n    return dict(quote_rmse_bp=float(np.sqrt(np.mean(abs(quoted)**2))*1e4),\n                heldout_rmse_bp=float(np.sqrt(np.mean(abs(error)**2))*1e4),\n                coefficient_rmse=float(np.sqrt(np.mean((np.asarray(xi).reshape(problem[\'truth\'].shape)-problem[\'truth\'])**2))))\n\ndef black_call(k,T,vol):\n    root=vol*np.sqrt(T); d1=-np.log(k)/root+root/2\n    return ndtr(d1)-k*ndtr(d1-root)\n\ndef implied_vol(k,T,c):\n    if not np.isfinite(c) or c<=max(1-k,0)+1e-13 or c>=1-1e-13:return np.nan\n    try:return brentq(lambda v:black_call(k,T,v)-c,1e-5,5.)\n    except ValueError:return np.nan\n'}
ASSETS = {'supplement.tex': "\\documentclass[10pt]{article}\n\\usepackage[margin=20mm]{geometry}\n\\usepackage{amsmath,amssymb,amsthm,bm,graphicx,booktabs,array,float,microtype}\n\\usepackage[hidelinks]{hyperref}\n\\usepackage{cite}\n\\newcommand{\\R}{\\mathbb R}\n\\newcommand{\\diag}{\\operatorname{diag}}\n\\newcommand{\\norm}[1]{\\lVert #1\\rVert}\n\\newcommand{\\ket}[1]{\\lvert #1\\rangle}\n\\newcommand{\\lchsnode}{\\eta}\n\\newcommand{\\lchsshift}{\\theta}\n\\renewcommand{\\theequation}{S\\arabic{equation}}\n\\setlength{\\emergencystretch}{2em}\n\\title{Technical supplement:\\\\Quantum Calibration of Local-Volatility Surfaces\\\\from Sparse Option Quotes}\n\\author{Kentaro Hoshisashi\\\\ \\small Department of Computer Science, University College London}\\date{}\n\\begin{document}\\maketitle\nThe main paper presents the financial problem, resulting implied-volatility\nsurfaces and price comparisons. This supplement retains the discretisation,\nderivations, quantum construction and numerical diagnostics needed to reproduce\nand assess those claims. The new main-paper surfaces are evaluated at four\nactual maturities. The earlier numerical benchmark and its diagnostic figures\nare retained separately, with their original model and execution provenance.\n\\tableofcontents\n\\input{measured_supplement.tex}\n\\section{Archived financial calibration problem}\n\\subsection{Financial normalisation and parameterisation}\nWith deterministic rates and proportional dividend yield, let $D(T)$ and $F(T)$ be the discount factor and forward. Write $k=K/F(T)$ and $c(T,k)=C(T,K)/(D(T)F(T))$. The forward-normalised equation is\n\\begin{equation}\n \\partial_T c=\\tfrac12 a(T,k)k^2\\partial_{kk}c,\\quad\n c(0,k)=(1-k)^+ . \\label{eq:dupire}\n\\end{equation}\nThe coefficient $a$ is the local variance evaluated at the corresponding forward-scaled asset price. The zero-carry case is included. Discrete cash dividends, American exercise, stochastic rates, and inconsistent forwards fall outside this model.\n\nOn slice $(T_{s-1},T_s]$, set $x=\\log k$ and\n\\begin{equation}\n a_s(x;\\xi_s)=a_{\\min}+\\sum_{j=1}^M\\xi_{sj}\\phi_j(x),\\qquad 0\\leq\\xi_{sj}\\leq1, \\label{eq:basis}\n\\end{equation}\nwhere $a_{\\min}=0.01$ and\n$\\phi_j(x)=0.04[1+\\cos(\\pi(x-\\mu_j)/w)]\\mathbf1_{|x-\\mu_j|\\leq w}$.\nThe archived financial model uses $M=4$, centres $(-0.6,-0.2,0.2,0.6)$ and $w=0.45$. This is a restricted positive family, not an arbitrary local-variance surface. Compact support makes these functions $C^1$ but generally not $C^2$ at support endpoints. They are not globally finite Fourier polynomials.\n\n\\subsection{Boundary lift, observations, and loss}\nUse $k\\in[k_0,k_{N+1}]=[e^{-1.2},e^{1.2}]$, with fixed boundary values $c(T,k_0)=1-k_0$ and $c(T,k_{N+1})=0$. These are a domain truncation assumption, not exact far-field values at finite strikes. Let $G(k)$ be the affine-in-strike interpolant of these boundary values. The interior homogeneous state is $v=c-G$. Since $G''=0$, the fitted discretisation below has no lift forcing.\n\nFor an off-grid quote, $W_s$ contains the two nonnegative piecewise-linear weights in \\emph{strike}. Boundary contributions are included before setting $y_s=c_s^{\\rm obs}-W_s^{\\rm full}G$. Thus $W_s$ need not be a row-selection matrix and $W_s^TW_s$ need not be a projector. The slice loss is\n\\begin{equation}\n \\mathcal L_s(\\xi_s)=\\frac1{q_s}\\norm{\\Omega_s(W_s v_s-y_s)}_2^2\n +\\lambda\\frac{\\norm{D_2\\xi_s}_2^2}{M-2}, \\label{eq:loss}\n\\end{equation}\nwith the last term omitted for $M\\leq2$. The synthetic solver comparison uses $\\Omega_s=I$ and $\\lambda=0$. Market quote weights are inverse half-spreads with a declared floor. A bounded L-BFGS-B solve fits slices sequentially, propagating the fitted preceding state. This bootstrap is not a joint optimum over all slices. Its order and accumulated error are retained in every comparison.\n\n\\section{Discrete arbitrage preservation}\nLet $x_i=x_0+ih$, $k_i=e^{x_i}$, with $N=2^n$ interior nodes. Define centred first and second log-grid differences $D_h,S_h$ and\n\\begin{equation}\n \\begin{gathered}\n Q_h=S_h-\\kappa_hD_h,\\qquad\\kappa_h=\\frac{2\\tanh(h/2)}h,\\\\\n \\rho_h=\\frac{2(\\cosh h-1)}{h^2}.\n \\end{gathered}\\label{eq:fitted}\n\\end{equation}\nFor $d_i^-=k_i-k_{i-1}$ and $d_i^+=k_{i+1}-k_i$, define the boundary-inclusive curvature operator\n\\begin{equation}\n (Bc)_i=\\frac{2}{d_i^-+d_i^+}\n \\left(\\frac{c_{i+1}-c_i}{d_i^+}-\\frac{c_i-c_{i-1}}{d_i^-}\\right). \\label{eq:B}\n\\end{equation}\nDirect coefficient comparison yields\n\\begin{equation}\n Q_h=\\rho_h\\diag(k_i^2)B. \\label{eq:identity}\n\\end{equation}\nIn particular, $Q_h$ annihilates constants and $k_i$, so it annihilates the lift exactly. Moreover, $\\kappa_h=1+O(h^2)$ and $\\rho_h=1+O(h^2)$: fitting changes the finite-grid operator while retaining second-order consistency with $\\partial_{xx}-\\partial_x$.\n\n\\subsection{Exact discrete surface}\nLet $a_i>0$ be constant within each time slice, with the fixed boundaries above. Start from the sampled convex payoff, and propagate the fitted semidiscrete equation exactly. At every nonnegative time, the full nodal surface is convex in strike and nondecreasing in maturity. It satisfies $(1-k_i)^+\\leq c_i\\leq1$ and every adjacent strike slope lies in $[-1,0]$.\n\n\\paragraph{Derivation.}\nLet $R$ inject interior vectors into full vectors, with zeros at the two boundary nodes, and put $E=\\frac{\\rho_h}{2}\\diag(a_i k_i^2)$. The full equation is $\\dot c=REBc$. For $q=Bc$, $\\dot q=BREq$. The matrix $BRE$ is Metzler because the off-diagonal entries of $BR$ are nonnegative and $E$ is positive diagonal. Hence $q(t)\\geq0$ from $q(0)\\geq0$, and $\\dot c=REq\\geq0$. Positivity continues across slice changes. Calendar monotonicity gives the payoff lower bound. Discrete comparison with the constant supersolution $1$ gives the upper bound. Convexity orders the adjacent slopes. The payoff lower bound at the left boundary gives the first slope at least $-1$. Nonnegativity and the zero right boundary give the last slope at most $0$. All intermediate slopes therefore lie in $[-1,0]$.\n\n\nPiecewise-linear interpolation in strike preserves these inequalities throughout the domain. Linear interpolation between two ordered convex maturity slices also preserves convexity and calendar monotonicity, though it is not an exact PDE solution between those slices. Extending by $1-k$ to the left and $0$ to the right preserves slope ordering. The extensions encode the truncated model. They do not establish accuracy outside the calibrated domain.\n\nWriting $L_h=-Q_h|_{\\rm interior}$ gives the design-affine homogeneous generator\n\\begin{equation}\n \\dot v=-A(\\xi)v,\\quad\n A(\\xi)=\\tfrac12\\diag(a(\\xi))L_h=A_0+\\sum_j\\xi_j A_j. \\label{eq:A}\n\\end{equation}\nBackward Euler also preserves the same properties because the corresponding resolvent of the Metzler curvature generator is nonnegative. An arbitrary truncated LCHS sum need not do so.\n\n\\subsection{Approximate-price bounds}\nLet $c_s$ be the exact nodal surface described above and $\\tilde c_s$ an approximation with $\\norm{\\tilde c_s-c_s}_\\infty\\leq\\delta_s$, using the same initial payoff ($\\delta_0=0$). Define the price butterfly\n$b_i(c)=(d_i^-c_{i+1}+d_i^+c_{i-1})/(d_i^-+d_i^+)-c_i$.\nThen\n\\begin{align}\n b_i(\\tilde c_s)&\\geq-2\\delta_s, \\label{eq:bfbound}\\\\\n \\tilde c_{s,i}-\\tilde c_{s-1,i}&\\geq-(\\delta_s+\\delta_{s-1}),\\\\\n (B\\tilde c_s)_i&\\geq-\\frac{4\\delta_s}{d_i^-d_i^+}. \\label{eq:denbound}\n\\end{align}\n\n\\paragraph{Derivation.}\nThe butterfly row has absolute coefficient sum $2$. Apply the triangle inequality to the price errors and to the two maturity errors. Finally, $(Bc)_i=2b_i(c)/(d_i^-d_i^+)$.\n\n\nThe derivative bound deteriorates as the grid is refined. A price-space tolerance cannot be reused as a density tolerance. Our diagnostic measures $\\delta_s$ against exact propagation at the \\emph{same fitted coefficients}. This tests a conditional perturbation bound, not an independently certified a priori solver tolerance. Raw approximate prices and exact re-evaluations are audited separately. The audits use unprojected, unclipped prices.\n\n\\section{LCHS and coherent objective construction}\n\\subsection{Accretivity and comparable quadrature}\nThe Hermitian part of $A$ can have a negative eigenvalue. Define a design-independent valid shift\n\\begin{equation}\n \\lchsshift=\\sum_{j=0}^{M}\\max\\{0,-\\lambda_{\\min}((A_j+A_j^T)/2)\\}.\n\\end{equation}\nSince $0\\leq\\xi_j\\leq1$, $A+\\lchsshift I$ has nonnegative Hermitian part for every allowed design. Write $A+\\lchsshift I=L+iH$, with $L\\succeq0$ and $H=H^\\dagger$. The LCHS representation~\\cite{lchs,nearoptimal} is approximated by\n\\begin{equation}\n e^{-tA}\\approx e^{\\lchsshift t}\\sum_{\\ell=1}^J w_\\ell\n e^{-it(H+\\lchsnode_\\ell L)}. \\label{eq:lchs}\n\\end{equation}\nWe use symmetric midpoint quadrature on $[-K_c,K_c]$. The simple kernel has density $1/[\\pi(1+\\lchsnode^2)]$. The improved kernel uses\n\\begin{equation}\n \\frac{f_\\beta(\\lchsnode)}{1-i\\lchsnode},\\quad\n f_\\beta(\\lchsnode)=\\frac{\\exp(2^\\beta-(1+i\\lchsnode)^\\beta)}{2\\pi},\\quad\\beta=0.8.\n\\end{equation}\nWeights are normalised to sum to one. The pre-normalisation defect and $\\alpha_{\\rm LCU}=\\sum|w_\\ell|$ are reported. Normalisation enforces the initial identity but does not restore the omitted tail. We jointly vary $J$ and $K_c$, reporting accuracy, $\\Delta\\lchsnode=2K_c/J$, warm evaluation time, shift amplification, and the evolution-scale proxy $K_c t\\norm A_2$. Equal $J$ alone is not equal Hamiltonian-simulation cost. A cutoff envelope in a figure is a diagnostic selected against a reference, not a calibration tuning rule.\n\nThe uniform shift also avoids differentiating an eigenvalue-dependent clipping rule. Automatic differentiation of the finite quadrature uses the Fr\\'echet derivative of a Hermitian exponential, with its divided difference evaluated by a stable sinc expression at coincident eigenvalues. Centred finite differences validate the implemented loss gradient. The sparse baseline uses its own exact discrete adjoint.\n\n\\subsection{Residual and objective block-encoding}\nFor one slice, absorb weighting and the factor $q_s^{-1/2}$ into $W,y$ if desired. Let a unitary dilation encode $W/\\gamma_W$, where $\\gamma_W\\geq\\norm W_2$. The small circuit prepares the initial homogeneous state, an LCU kernel register, and a design-controlled SELECT of propagators. It then unprepares the kernel and applies the quote dilation. Its successful forward branch has amplitude $W\\tilde P(\\xi)v_0/a$, where\n\\begin{equation}\n a=\\norm{v_0}_2e^{\\lchsshift t}\\alpha_{\\rm LCU}\\gamma_W.\n\\end{equation}\nCombining that branch with a prepared reference vector $y$ using relative weights $a$ and $\\norm y_2$ produces a residual unitary $V$ with success probability\n\\begin{equation}\n p(\\xi)=\\frac{\\norm{W\\tilde P(\\xi)v_0-y}_2^2}{s_0^2},\n \\qquad s_0=a+\\norm y_2. \\label{eq:prob}\n\\end{equation}\nFor the success projector $\\Pi$, define $U=V^\\dagger(2\\Pi-I)V$. A flag-qubit Hadamard construction encodes $(I+U)/2$. Its clean-workspace block equals the diagonal objective $p(\\xi)$. We verify both the residual probability and this amplitude for a superposition of all design points. No precomputed objective value is loaded into the circuit.\n\nThe demonstrator has two coefficient parameters and one maturity. It enumerates the small design grid to construct dense controlled unitaries. Transpilation reports qubits, depth, and single-qubit/CNOT counts with all-to-all connectivity. This construction is explicit but expensive. It is not the efficient arithmetic SELECT assumed in asymptotic analyses. State preparation, norm factors, data access, postselection/amplification, and readout cannot be treated as free. Extending to a coherent multi-slice bootstrap requires retaining preceding design and state dependence. The classical calibration bootstrap does not implement that extension.\n\n\\subsection{Precision, smoothness, and optimisation scope}\nWith $m$ bits per coefficient, $h_\\xi=(2^m-1)^{-1}$ and $mM$ design qubits are needed per slice. Nearest-grid rounding gives $\\norm{\\Delta\\xi}_\\infty\\leq h_\\xi/2$ and\n\\begin{equation}\n \\norm{\\Delta A}_2\\leq\\frac{h_\\xi}{2}\\sum_j\\norm{A_j}_2.\n\\end{equation}\nThe notebook re-evaluates rounded designs for $m=2,4,6,8,10,12$. This is a discretisation sensitivity experiment, not discrete re-optimisation.\n\nIf a Fourier approximation satisfies $\\norm{\\tilde\\phi_j-\\phi_j}_\\infty\\leq\\tau$, then\n\\begin{equation}\n \\norm{\\tilde a-a}_\\infty\\leq M\\tau,\\qquad\n \\norm{\\tilde A-A}_2\\leq\\tfrac12M\\tau\\norm{L_h}_2.\n\\end{equation}\nPositivity is guaranteed by $M\\tau<a_{\\min}$, although an individual truncated basis may be negative. The original pointwise basis is used in calibration. The Fourier diagnostic samples one of the four basis functions on 4096 points. It does not certify a continuous supremum or a simultaneous bound for all basis functions. Its reported budget is therefore illustrative, whereas the inequalities above are conditional on true supremum bounds. Analytic-function polylogarithmic approximation claims~\\cite{yano} cannot simply be transferred to this compactly supported $C^1$ family.\n\nWe simulate a QHD-inspired Hamiltonian schedule~\\cite{qhd}, $e^{-s}H_{\\rm kin}+e^s\\widehat F$, by a symmetric split step on the \\emph{classically tabulated} verified finite design landscape. Exact enumeration supplies the finite-grid reference. Normalisation, probability near that grid minimum, and time-step refinement are reported. This is separate from the coherent objective test. Strong convexity, continuum global optimality, and an end-to-end quantum speedup are not established. Published complexity results~\\cite{sato} require their own conditioning, smoothness, and oracle assumptions. Here $\\norm{L_h}=O(h^{-2})$, shift/norm factors, design precision, and readout all remain visible costs.\n\n\n\n\\section{Matched pricing-solver comparison}\n\\begin{figure}[H]\\centering\n\\includegraphics[width=.88\\linewidth]{figures/solver_comparison.pdf}\n\\caption{Changing only the forward solver, $N=64$, out-of-family target with\n5 bp price noise. Price RMSE is averaged within each dataset across five\nstarts, then across three datasets. Bars show dataset sample SD. CPU\ncalibration times exclude compilation and subsequent evaluation.}\n\\end{figure}\n\\section{Four-maturity smile and skew examples}\nThe main surface figure uses the positive quadratic family\n\\[\n a_s(x)=0.01+0.08\\xi_{s1}+0.1\\xi_{s2}(x+0.45)^2\n                   +0.1\\xi_{s3}(x-0.45)^2.\n\\]\nThe smile coefficients are $(0.10,0.50,0.50)$, $(0.16,0.50,0.50)$,\n$(0.22,0.50,0.50)$ and $(0.28,0.50,0.50)$. The skew coefficients are\n$(0.20,0.10,0.85)$, $(0.22,0.12,0.80)$, $(0.24,0.14,0.75)$ and\n$(0.26,0.16,0.70)$. Intervals end at 0.25, 0.5, 0.75 and 1 year.\nEach expiry contains quotes at normalised strikes\n$0.70,0.78,0.87,0.95,1,1.06,1.15,1.28,1.40$.\nThese controlled examples illustrate two familiar shapes. They are neither\nmarket observations nor additional independently measured quantum calibrations.\n\nExact 1,024-node propagation generates the reference prices. Sequential bounded\nleast squares fits three coefficients per interval on 128 nodes. Exact,\nbackward-Euler (1,024 substeps per interval) and beta-LCHS evaluations then use\nthe same fitted coefficients. LCHS uses 8,192 terms and cutoff 96. The explicit\ntridiagonal spectral implementation is checked against dense matrix exponentials\non a small grid. Reference refinement is recorded in the validation JSON.\nThe two forward approximations are not separately refitted in this example.\n\nAll four maturity rows are independently propagated. The 3D mesh connects\nthese computed rows for display. The one-year panels compare the fine-grid\ntarget, exact coarse-grid model, and LCHS model against the actual quotes.\nThe plotted price and IV arrays are exported to\n\\path{surface_examples/surface_plot_data.csv}. No price clipping or IV smoothing\nis applied. Held-out price RMSE excludes the nine calibration strikes.\n\n\\section{Archived benchmark and additional diagnostics}\nThe supplementary benchmark figures retain the original four-cosine family.\nTheir numerical predictions and derivative checks are unchanged by the new\nquadratic illustrative examples. Where an archived display interpolates between\nits two quoted maturities, that display is not an additional calibration.\n\nThis supplement audits the supplied completed run \\texttt{paper\\_9684a7f0bec7}.\nNo original calibration was rerun. All 568 planned keys are accounted for.\n567 completed runs and one failed AH input are retained. Errors below are\nnormalised prices multiplied by $10^4$ when labelled bp. Dataset standard\ndeviations are descriptive, not confidence intervals. Five initialisations\nwithin one dataset do not constitute five independent market datasets.\n\n\\section{Comparisons and the missing AH input}\n\\input{tables/all_cases.tex}\nIn-family dataset 303 at noise level 5 bp contains a first-maturity call at\n$k=0.7$ with observed value approximately 0.299932374, below intrinsic value 0.3.\nBlack implied volatility is undefined for this input and the AH baseline\nstops. The quote is not clipped. Comparisons on this input do not have the\nsame feasibility conditions as the other datasets. The all-case table makes\nthe denominator difference explicit. The following table instead pairs the\nsame two feasible datasets for every method.\n\\input{tables/paired_in_family.tex}\n\\begin{figure}[H]\\centering\\includegraphics[width=.65\\linewidth]{figures/heldout_comparison.pdf}\n\\caption{Out-of-family noisy case: means and sample SD across three independent\ndata seeds, after averaging five initialisations (AH deterministic).}\\end{figure}\n\n\\section{Finite approximation and diagnostic scope}\nAll 240 finite-LCHS calibrations fail the full-domain butterfly and calendar\ntests at price tolerance $10^{-8}$. For beta LCHS, only the $N=128$ butterfly\ntest also fails within the quote/test band (15 of 40 runs). Exact re-evaluation\npasses at all 240 fitted designs. Full-grid details, maxima and band counts\nare in \\path{audit/arbitrage_by_method_grid.csv}. Per-run independently\nreconstructed metrics are in \\path{audit/independent_checks.csv}.\nBoth comparison kernels use $J=2048$ in calibration, but their cutoffs differ\n(beta 48; simple 64). The forward sweep explicitly uses 16, 32 and 64.\nThe simple kernel has 4764 invalid held-out model IV evaluations among 43080.\nIts conditional valid-IV RMSE must be accompanied by this coverage.\nAll reported ``certificate passes'' use measured error against exact\npropagation at the same coefficients and are conditional perturbation tests.\n\n\\section{Quantisation, Fourier approximation and identifiability}\n\\begin{figure}[H]\\centering\\includegraphics[width=.78\\linewidth]{figures/precision_basis.pdf}\n\\caption{One-fit coefficient rounding and sampled Fourier diagnostics. The\nFourier panel evaluates one of four compactly supported basis functions on\n4096 points. Its sampled maximum and derived illustrative budgets are not\ncontinuous supremum certificates for all basis functions. No Fourier\napproximation is used in the actual calibration.}\\end{figure}\nAt four bits, training and held-out RMSE are 13.711 and 12.600 bp. The continuous\ntraining fit is 0.693 bp. At ten and twelve bits, training errors are 0.696 and\n0.693 bp. This is rounding without discrete re-optimisation on one fitted case.\nThe tested high-degree Fourier decay is consistent with algebraic convergence.\nNeither an asymptotic rate proof nor polylogarithmic complexity follows.\n\\begin{figure}[H]\\centering\\includegraphics[width=.78\\linewidth]{figures/fd_identifiability.pdf}\n\\caption{A fixed-design backward-Euler refinement diagnostic and local quote\nJacobian singular values for one selected exact fit.}\\end{figure}\nThe maximum FD price error falls from $2.0279\\times10^{-4}$ at 32 steps to\n$6.3543\\times10^{-6}$ at 1024 and $3.1773\\times10^{-6}$ at 2048. This is a\nfixed-design convergence check, not a recalibration at every time step count.\nJacobian condition numbers are about 467 and 59 for the two slices.\n\n\\section{Sensitivity experiments and confounders}\n\\begin{figure}[H]\\centering\\includegraphics[width=.78\\linewidth]{figures/ablation.pdf}\n\\caption{Basis-family and regularisation sensitivity on one noisy out-of-family\ndataset and one initialisation. Centres and widths change with $M$, so the\nbases are not nested. The second-difference penalty is absent for $M=2$ by\ndefinition. This is not a validation-selected performance estimate.}\\end{figure}\nAt $M=8$, $\\lambda=10^{-7}$, exact and beta held-out errors are about 2.50 and\n2.77 bp. A prospective model comparison must select $M,\\lambda$ using training\nand validation information and assess a fresh test set. The present inspected\nheld-out set cannot serve as an untouched confirmatory test for that choice.\n\\begin{figure}[H]\\centering\\includegraphics[width=.78\\linewidth]{figures/domain_slices.pdf}\n\\caption{Domain and slice sensitivities. Fixed spatial node count means that\nchanging the domain changes spacing. Nominal 2, 4 and 8 slice experiments\ninsert the true change point, giving 3, 5 and 9 actual slices and 21, 35 and\n63 quotes. They change both model/time resolution and available observations.}\\end{figure}\n\n\\section{Neural and quantum diagnostics}\n\\begin{figure}[H]\\centering\\includegraphics[width=.78\\linewidth]{figures/pinn_losses.pdf}\n\\caption{Fixed/adaptive PINN training diagnostics. The comparator is a\ndocumented adaptation with a fixed 6000-epoch budget. It is not an exact\nreproduction of the original Whack-a-Mole implementation.}\\end{figure}\nNeither neural model has a counted common-grid butterfly/calendar failure in\nits 40 completed runs. This sampled result is not a structural guarantee over\nall strikes and maturities. Fixed-epoch completion is not optimiser convergence.\n\\begin{figure}[H]\\centering\\includegraphics[width=.78\\linewidth]{figures/qhd_diagnostics.pdf}\n\\caption{QHD-inspired split-step evolution on a classically tabulated 256-point\nlandscape, with a separate 1500-versus-3000-step refinement check. The coherent\ncircuit evaluates only four corner designs.}\\end{figure}\nThe near-minimum criterion is within one per cent of the finite loss range.\nProbability rises from 13.28\\% to 76.92\\%. The refined distribution differs in\ntotal variation by $8.46\\times10^{-7}$. These observations do not imply\ncontinuous global optimality or quantum advantage.\n\n\\section{Reproduction and provenance}\nThe companion \\texttt{surface\\_calibration\\_experiments.ipynb} runs the new\nfour-maturity surface examples and measured calibration, checks circuits and regenerates figures and\nTeX. It embeds the immutable archived evidence and verifies its hash before\nuse. The independent audit reads saved predictions without executing code\nfrom that evidence. The optional original full experiment notebook reruns\nthe larger numerical benchmark. These execution scopes are kept separate.\nOriginal raw observations, checkpoints, CSV diagnostics and source code are\nretained under \\texttt{paper/evidence/} in the repository\n\\url{https://github.com/khoshisashi/quantum-local-vol}. No market observations\nwere supplied. Runtime versions and CPU backend are in the original manifest.\nThe submitted input ZIP has SHA-256\n\\par\\noindent\\texttt{6684a5cf912b9703b9180395bb1df48e832}\\newline\n\\texttt{fcc5944610421d279a211952b4fca} (concatenate the two lines).\n\n\\footnotesize\n\\input{supplement_references.tex}\n\\end{document}\n", 'measured_supplement.tex': "\\section{New measured multi-expiry experiment}\nThe main paper adds a measured hybrid calibration to the larger archived\nfinancial benchmark. These are separate experiments. The new circuit has\n$N=4$ interior nodes, two basis functions with centres $(-0.55,0.55)$ and\nwidth $0.85$, and maturities $(0.5,1)$ years. Four irregular quotes per maturity\nare used. The exact same-grid target has coefficients $(0.75,0.25)$ and\n$(0.25,0.75)$. It isolates circuit, quadrature and sampling errors. The larger\narchived benchmark instead uses $M=4$ and fine-grid target generation.\n\n\\subsection{Product of successful propagation blocks}\nLet $U_s$ have successful block $\\widetilde P_s/(e^{\\lchsshift\\Delta_s}\\alpha)$.\nAllocate a fresh LCU register to every interval and use the same system register.\nSince subsequent operators do not act on a previous interval's LCU register,\nprojection of all of them onto zero gives\n\\begin{equation}\n \\left(\\prod_{r=1}^s\\langle0|_{a_r}\\right)U_s\\cdots U_1\n \\left(\\prod_{r=1}^s|0\\rangle_{a_r}\\right)\n =\\frac{\\widetilde P_s\\cdots\\widetilde P_1}{e^{\\lchsshift T_s}\\alpha^s}.\n\\end{equation}\nFailed branches cannot re-enter a previously successful subspace, since that\nregister is never reused. There is no intermediate measurement or replacement\nwith a classically computed state.\n\n\\subsection{Observation and residual measurement}\nPad the quote map to an $N\\times N$ matrix $W_s$ and its target to an\n$N$-vector $y_s$. This prototype requires $q_s\\leq N$.\nFor $B=W_s/\\gamma_s$, $\\gamma_s=\\max(1,\\norm{W_s}_2)$, the circuit uses\n\\begin{equation}\n U_B=\\begin{pmatrix}B&\\sqrt{I-BB^\\dagger}\\\\\n \\sqrt{I-B^\\dagger B}&-B^\\dagger\\end{pmatrix}.\n\\end{equation}\nThe implementation checks unitarity numerically. With\n$a_s=\\norm{v_0}_2e^{\\lchsshift T_s}\\alpha^s\\gamma_s$ and\n$d_s=\\norm{y_s}_2$, prepare the residual branch with amplitudes\n$\\sqrt{a_s/(a_s+d_s)}$ and $\\sqrt{d_s/(a_s+d_s)}$.\nControlled preparation of the propagated state and $y_s/d_s$, a relative\nminus sign and inverse branch preparation produce successful residual\n$(W_s\\widetilde v_s-y_s)/(a_s+d_s)$.\nThe all-zero ancillary frequency therefore estimates the squared residual,\nwith its known scale retained. The supplied financial target has $d_s>0$.\nA general zero-target instance would need its trivial separate branch.\n\n\\subsection{Search, replication and interval reporting}\nBoth matrix and measured objectives use the same bounded pattern search.\nStarting at four coefficients equal to $0.5$, two sweeps test the incumbent\nand positive/negative steps of $0.25$ along four coordinate directions and\ntwo within-interval tilt directions. Coordinates are clipped to $[0,1]$ and\nduplicate candidates are removed. Ties retain the earlier candidate, with\nthe incumbent tested first. The finite search does not claim global optimisation.\n\nThe paper configuration uses $J=8$, cutoff $12$, and respectively $2^{18}$\nand $2^{22}$ repetitions per maturity per candidate, each with seeds\n$101,202,303$. Only backend finite-shot counts enter coefficient decisions.\nIndependent matrix and statevector checks occur outside the search.\nFinal smile plots are matrix re-evaluations at the selected coefficients.\nThe raw measurement record separately shows what was observed. Held-out\nerrors use the known synthetic target only for reporting.\n\nAfter selection, a fresh evaluation supplies the reported interval. Each\nmaturity uses a Clopper--Pearson binomial interval with tail probabilities\n$0.05/(2S)$. Scaling and averaging these intervals gives a conservative\n95\\% interval for the final finite-circuit objective by a union bound.\nFor zero observed successes, the upper endpoint remains positive. Per-call\nWald standard errors are also logged as descriptive approximations, but are\nnot used to certify a zero loss. Calibration and independent-check shot/gate\ntotals have separate fields. Transpiled $u$/CNOT circuits are exported\nin QPY format. Counts assume all-to-all connectivity without error correction.\n\nThe resource sweep uses cutoff $4$ for $J\\leq8$ and $32$ otherwise, varying\n$N,J$ and the number of maturity intervals. This is a gate-size diagnostic.\nThe $J=8$ calibration uses cutoff $12$. Both cutoffs are recorded explicitly.\nAn additional matrix-only quadrature sweep varies cutoff and term count\ntogether. Its accuracy figures are not hardware measurements.\n", 'supplement_references.tex': "\\begin{thebibliography}{99}\n\\setlength{\\itemsep}{0pt}\n\\bibitem{lchs} D. An, J.-P. Liu, and L. Lin, ``Linear combination of Hamiltonian simulation for nonunitary dynamics with optimal state preparation cost,'' \\emph{Physical Review Letters}, vol. 131, no. 15, Art. no. 150603, 2023, doi: \\href{https://doi.org/10.1103/PhysRevLett.131.150603}{10.1103/PhysRevLett.131.150603}.\n\\bibitem{nearoptimal} D. An, A. M. Childs, and L. Lin, ``Quantum algorithm for linear non-unitary dynamics with near-optimal dependence on all parameters,'' \\emph{Communications in Mathematical Physics}, vol. 407, Art. no. 19, 2026, doi: \\href{https://doi.org/10.1007/s00220-025-05509-w}{10.1007/s00220-025-05509-w}.\n\\bibitem{yano} H. Yano and Y. Sato, ``Quantum framework for parameterizing partial differential equations via diagonal block-encoding,'' arXiv preprint \\href{https://arxiv.org/abs/2603.01358v1}{arXiv:2603.01358v1}, 2026.\n\\bibitem{qhd} J. Leng, E. Hickman, J. Li, and X. Wu, ``Quantum Hamiltonian descent,'' arXiv preprint \\href{https://arxiv.org/abs/2303.01471v1}{arXiv:2303.01471v1}, 2023.\n\\bibitem{sato} Y. Sato, J. Kato, H. Yano, K. Ito, and N. Yamamoto, ``Explicit block-encoding for partial differential equation-constrained optimization,'' arXiv preprint \\href{https://arxiv.org/abs/2511.14420v2}{arXiv:2511.14420v2}, 2026.\n\\end{thebibliography}\n", 'requirements.txt': 'numpy==2.3.5\nscipy==1.17.0\npandas==2.2.3\nmatplotlib==3.10.8\nqiskit==2.5.2\nqiskit-aer==0.17.2\npypdf==5.9.0\nnbformat==5.11.1\nipython==9.17.1\n'}
EVIDENCE_SHA256 = '8856d868e87503ee99d6cc9c5299ed1aa520fa097e11280c331fd81ebc952f9f'
EVIDENCE_BASE64 = '...'
LEGACY_NOTEBOOK_ZLIB64 = '...'
import base64, hashlib, io, json, zipfile, shutil, zlib
from pathlib import Path
root=Path(OUTPUT_ROOT).resolve();root.mkdir(parents=True,exist_ok=True)
src=root/"src";src.mkdir(exist_ok=True)
paper=root/"paper";paper.mkdir(exist_ok=True)
run=root/"new_experiment";run.mkdir(exist_ok=True)
for name,content in SOURCES.items(): (src/name).write_text(content,encoding="utf-8")
for name,content in ASSETS.items():
    path=paper/name;path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(content,encoding="utf-8")
blob=base64.b64decode(EVIDENCE_BASE64)
assert hashlib.sha256(blob).hexdigest()==EVIDENCE_SHA256
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    for info in z.infolist():
        path=(paper/info.filename).resolve()
        if not path.is_relative_to(paper): raise ValueError("Archive path escapes output directory")
        if not info.is_dir():
            path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(z.read(info))
(root/"optional").mkdir(exist_ok=True)
(root/"optional/full_legacy_experiment.ipynb").write_bytes(zlib.decompress(base64.b64decode(LEGACY_NOTEBOOK_ZLIB64)))
sys.path.insert(0,str(src))
from IPython.display import display, Image, FileLink
print("Source modules:",list(SOURCES));print("Archived benchmark:",EVIDENCE_SHA256)


## 2. Set up the pricing model and check the circuits independently

Four quotes are placed at each of two maturities (six months and one year). The generating coefficients are known but are not passed to the measured search.
The circuit of the later maturity also passes through the propagation up to the earlier one.

Only here is the circuit statevector compared with an independent matrix calculation. **No statevector is used to decide the coefficients of the calibration.**


In [ ]:
import numpy as np, pandas as pd
from surface_model import grid, data_for, save_json
from quantum_experiments import circuit_checks, run_calibration_experiment, resource_sweep
problem=data_for(grid())
checks=circuit_checks(problem,terms=4,cutoff=4.,out=run)
display(checks[["expiry","probability","expected","absolute_error"]])
assert checks.absolute_error.max()<2e-10


## 3. Estimate the surface coefficients from measurements

For every candidate the circuit is decomposed into gates, and the number of times the ancillas are found in the success state is counted. Only the loss obtained by scaling these counts with known factors is passed to the classical bounded pattern search.
The same search order is used for exact propagation, finite differences and the matrix of the same finite LCHS. The unit of comparison is an independent sampler seed.

A checkpoint is saved after every run. The measurements used by the search and the final independent check are kept apart in the records.


In [ ]:
results=run_calibration_experiment(run,PROFILE)
display(results[["method","shots","seed","evaluations","quote_rmse_bp","heldout_rmse_bp","coefficient_rmse"]])
status=json.loads((run/"status.json").read_text())
assert status["completed"] and status["no_statevector_access_in_optimiser"]


## 4. Circuit resources and approximation accuracy

The numbers of qubits and CNOT gates, the depth and the classical synthesis time of the decomposed circuits are saved.
CNOT counts are per circuit execution. The totals multiplied by the number of shots are in the calibration CSV.
The resource sweep is a diagnostic of gate counts, so its cutoff differs from that of the calibration in some cases, and the CSV records both.

The additional quadrature sweep is a **matrix calculation**. The number of terms and the cutoff are varied independently to show how the choice of approximation affects the price accuracy.


In [ ]:
resources=resource_sweep(run,PROFILE)
from experiment_figures import quadrature_diagnostic
quadrature=quadrature_diagnostic(run)
display(resources[["N","J","cutoff","expiries","qubits","cx","depth"]])
display(quadrature[["N","J","cutoff","heldout_rmse_bp"]].head(10))


## 5. Smile, skew and the diagram of the calibration

From nine quotes at each of four maturities, the coefficients of three positive quadratic basis functions are calibrated interval by interval. Two sets of generating coefficients are stated, a smile and a skew.
Reference prices use 1,024 nodes; calibration and evaluation use 128. With the **same coefficients**, calibrated by exact propagation, the surface is evaluated by exact propagation, finite differences and LCHS.
The example compares the representation of the surface and the forward approximations. It is not an independent calibration from quantum measurements.
LCHS uses 8,192 terms and cutoff 96; `smoke` uses a lighter setting.

Each row is the smile or the skew: the surface over the four maturities with the input quotes on the left, the one-year slice on the right. The axes are k, T and IV (%). IV is not smoothed, and invalid prices are not clipped.


In [ ]:
import surface_example, submission_figures
examples=root/"surface_examples"
example_accuracy=surface_example.run(examples,"paper" if PROFILE=="paper" else "smoke")
example_checks=surface_example.validate_reference(examples,"paper" if PROFILE=="paper" else "smoke")
figure_values=submission_figures.generate(examples,run,paper)
display(Image(filename=str(paper/"figures/calibration_architecture.png")))
display(Image(filename=str(paper/"figures/financial_surfaces.png")))
display(example_accuracy[["case","method","heldout_rmse_bp","quote_rmse_bp"]])


## 6. Measurement precision and calibration results

The figure shows how finely the final price residual can be resolved as the number of shots grows.
Points and 95% intervals are values **measured independently after the search**, at the selected coefficients. Crosses are the values of the finite-LCHS matrix at the same coefficients.
All declared seeds are shown, not the best one. A count of zero does not establish that the true residual is zero.
The intervals are neither confidence intervals of the coefficients nor a guarantee of global optimality.


In [ ]:
display(Image(filename=str(paper/"figures/measurement_precision.png")))
measured=results[results.method=="quantum_counts"].copy()
display(results[["method","shots","seed","heldout_rmse_bp","coefficient_rmse"]])
display(measured[["shots","seed","independent_final_measured_loss","independent_final_mse_lower","independent_final_mse_upper","total_shots","total_cx_executions"]])


## 7. Audit the archived comparison with classical methods

The 567 completed runs and the one failed run of the archived benchmark are audited again. The calibrations of that benchmark are not rerun here.
LCHS, exact propagation and finite differences share one coefficient model; AH and PINN use different representations, and the two groups should not be confused.
The no-arbitrage violations of finite LCHS in the archived benchmark are reported as they are.

Fig. 4 compares, at the same coefficients, the first maturity of the clean outside-model case with N=128 and optimiser start 0.
This is the stress case with the largest butterfly violation inside the quoted band among the stored results, not an average case.
LCHS is recomputed with 2,048 and 4,096 terms (cutoff 48, beta 0.8) and compared with exact propagation and finite differences.
The top panel shows the difference from the exact price at the same coefficients, the bottom panel the butterfly price of three adjacent strikes. A negative value is a convexity violation.
This is neither a new calibration nor a new circuit measurement. The improvement inside the quoted band and the violations that remain outside it are reported separately.


In [ ]:
import audit_results
audit=audit_results.run(paper/"evidence",paper)
assert audit["completed_runs"]==567 and audit["failed_runs"]==1
summary=pd.read_csv(paper/"audit/summary_verified.csv")
a=summary[(summary.n==6)&(summary.truth=="out")&(summary.noise_bp==5)].copy()
a["heldout_price_bp"]=a.heldout*1e4;a["dataset_sd_bp"]=a.heldout_sd*1e4
display(a[["method","heldout_price_bp","dataset_sd_bp","opt_s","datasets","runs"]])
import consistency_example
consistency_dir=root/"consistency_example"
consistency_checks=consistency_example.run(paper/"evidence",consistency_dir)
submission_figures.consistency(consistency_dir,paper)
display(Image(filename=str(paper/"figures/financial_consistency.png")))
display(pd.DataFrame(consistency_checks["metrics"]).T)
finance={"surface_examples":figure_values,"validation":example_checks,
         "fixed_coefficient_consistency":consistency_checks,
         "scope":"new common-coefficient quadratic examples; retained independently calibrated benchmark"}


## 8. Collect the supplement, tables, figures and results

The technical supplement is compiled only when the six runs of the `paper` profile have completed.
Other profiles write their figures and logs but no PDF, so that their numbers are not taken for those of the paper.
On Colab without a TeX installation this cell installs the packages it needs.

The ZIP archive holds the figures and tables, the audited archived data, the new measurement logs, the runtime environment, the sources, the circuits in QPY format and the supplement.
`optional/full_legacy_experiment.ipynb` reruns the large archived experiment in full, with the classical solvers, AH and PINN (a long run).


In [ ]:
from datetime import datetime, timezone
import platform
paper_ready=(PROFILE=="paper" and status["measured_runs"]==6 and json.loads((examples/"configuration.json").read_text())["profile"]=="paper")
if not paper_ready:
    (paper/"supplement.pdf").unlink(missing_ok=True)
    print("Non-paper profile: figures and logs generated; the supplement is not compiled.")
elif BUILD_PDF:
    if shutil.which("pdflatex") is None and Path("/content").exists():
        subprocess.check_call(["apt-get","-qq","update"])
        subprocess.check_call(["apt-get","-qq","install","-y","texlive-latex-base","texlive-latex-extra","texlive-fonts-recommended"])
    if shutil.which("pdflatex"):
        from pypdf import PdfReader
        for suffix in ["aux","log","out","toc"]:
            (paper/("supplement."+suffix)).unlink(missing_ok=True)
        for _ in range(3):
            log=subprocess.run(["pdflatex","-interaction=nonstopmode","-halt-on-error","supplement.tex"],cwd=paper,text=True,capture_output=True)
            (paper/"supplement_compile.txt").write_text(log.stdout+log.stderr)
            if log.returncode:raise RuntimeError("TeX compilation failed; see supplement_compile.txt")
        generated_pdf=paper/"supplement.pdf"
        assert generated_pdf.read_bytes().rstrip().endswith(b"%%EOF"), "Incomplete PDF"
        pages=len(PdfReader(generated_pdf,strict=True).pages)
        assert pages>0, "Empty PDF"
        print("Compiled the technical supplement:",pages,"pages.")
    else:
        print("TeX files generated. Install LaTeX or compile paper/supplement.tex elsewhere.")
save_json(root/"reproduction_manifest.json",dict(profile=PROFILE,paper_ready=paper_ready,
    generated_utc=datetime.now(timezone.utc).isoformat(),python=platform.python_version(),
    original_archive_sha256="6684a5cf912b9703b9180395bb1df48e832fcc5944610421d279a211952b4fca",
    historical_evidence_sha256=EVIDENCE_SHA256,
    source_sha256={k:hashlib.sha256(v.encode()).hexdigest() for k,v in SOURCES.items()},
    new_experiment_executed=True,new_surface_examples_executed=True,
    fixed_coefficient_consistency_reexecuted=True,
    historical_benchmark_reexecuted=False,hardware_execution=False,
    financial_audit=audit,finance_figure_checks=finance))
archive=root.parent/("quantum_surface_"+PROFILE+"_outputs.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(root.rglob("*")):
        if p.is_file() and "__pycache__" not in p.parts and p.suffix not in {".aux",".log",".out",".toc"}:
            z.write(p,p.relative_to(root))
print("Results:",archive)
display(FileLink(str(archive)))
if paper_ready and (paper/"supplement.pdf").exists():display(FileLink(str(paper/"supplement.pdf")))


In [ ]:
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(archive))
